In [12]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

### Структура RAW-таблиц: схемы и типы полей

Смотрим состав колонок и типы данных в четырёх источниках, чтобы понять, какие признаки доступны для построения профиля игрока и какие поля потребуют приведения типов.

**Выводы:**
* `pg__matches` — **23 колонки**. Ключевые для сегментации: `game_mode`, `lobby_type`, `human_players`, `duration`, `start_time`, `radiant_win`. Служебные: `loaded_at`, `sys_insert_stamp`, `src_modified_stamp`, `job_id`.
* `pg__match_players` — **37 колонок**. Есть всё необходимое для признаков эффективности: `kda` (float), `gold_per_min`, `xp_per_min`, `hero_damage`, `tower_damage`, `denies`, `last_hits`, `net_worth`, `hero_healing`, `level`. Также предметы (`item_0..5`, `backpack_0..2`, `item_neutral`) и флаги Aghanim's.
* Поле `rank` присутствует в `match_players`, но его пригодность требует отдельной проверки (снапшот на дату парсинга vs на дату матча).
* Справочники `pg__game_modes` и `pg__lobby_types` — по 6 колонок, содержат флаг `balanced`, useful для отсева нестандартных режимов.
* Все числовые игровые показатели имеют тип `integer`, `kda` — `float`. Агрегаты через `AVG()` дадут `double`, а деление — `decimal`; при подготовке к модели потребуется единое приведение к `double`.
* Поля `barracks_status_*` и `tower_status_*` в `matches` — битовые маски состояния строений, для текущей задачи не используются.

In [13]:
TABLES = [
    "lakehouse.raw.pg__matches",
    "lakehouse.raw.pg__match_players",
    "lakehouse.raw.pg__game_modes",
    "lakehouse.raw.pg__lobby_types",
]

with SparkSessionContext('h1-step0-schema') as spark:
    for t in TABLES:
        df = spark.table(t)
        print("=" * 80)
        print(t)
        print("columns:", len(df.columns))
        print(", ".join(df.columns))
        print("-" * 80)
        df.printSchema()

lakehouse.raw.pg__matches
columns: 23
barracks_status_dire, barracks_status_radiant, cluster, dire_score, discover_run_id, duration, enrich_run_id, enrichment_attempts, first_blood_time, game_mode, human_players, loaded_at, lobby_type, match_id, match_seq_num, radiant_score, radiant_win, start_time, tower_status_dire, tower_status_radiant, job_id, sys_insert_stamp, src_modified_stamp
--------------------------------------------------------------------------------
root
 |-- barracks_status_dire: integer (nullable = true)
 |-- barracks_status_radiant: integer (nullable = true)
 |-- cluster: integer (nullable = true)
 |-- dire_score: integer (nullable = true)
 |-- discover_run_id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- enrich_run_id: integer (nullable = true)
 |-- enrichment_attempts: integer (nullable = true)
 |-- first_blood_time: integer (nullable = true)
 |-- game_mode: integer (nullable = true)
 |-- human_players: integer (nullable = true)
 |-- loaded

### Первый взгляд на данные: примеры строк и справочники

Смотрим реальные значения, чтобы проверить семантику полей и расшифровать коды режимов и лобби.

**Выводы:**
* В первых же строках `match_players` видны записи с `account_id = 4294967295` — это анонимные игроки (закрытый профиль). Таких строк много, они не позволяют построить историю аккаунта.
* У всех анонимов `rank = NULL`. Это подтверждает связь между приватностью профиля и отсутствием ранга.
* `player_slot` кодирует сторону: 0–4 — Radiant, 128–132 — Dire. Флаг `is_radiant` дублирует эту информацию.
* Пример матча `9045646600`: у игрока на hero_id=18 (Sven) 32 убийства, 801 GPM, 113 922 урона при net_worth 51 193 — потенциальная аномалия, но контекст матча неизвестен.
* Справочник режимов: `mode_id = 22` → `game_mode_all_draft` (`balanced = true`), `mode_id = 23` → `game_mode_turbo` (`balanced = false`). Turbo помечен как несбалансированный режим.
* Справочник лобби: `lobby_id = 7` → `lobby_type_ranked` (`balanced = true`), `lobby_id = 0` → `lobby_type_normal`. Целевой сегмент — комбинация 22 + 7.
* В справочниках присутствуют исторические и событийные режимы (Diretide, Greeviling, Ability Draft), которых в выборке матчей нет.

In [14]:
with SparkSessionContext('h1-step0-peek') as spark:
    (spark.table("lakehouse.raw.pg__matches")
        .select("match_id", "start_time", "duration", "game_mode", "lobby_type",
                "human_players", "radiant_win", "radiant_score", "dire_score", "cluster")
        .show(5, truncate=False))

    (spark.table("lakehouse.raw.pg__match_players")
        .select("match_id", "player_slot", "account_id", "hero_id", "is_radiant",
                "kills", "deaths", "assists", "kda", "gold_per_min", "xp_per_min",
                "hero_damage", "net_worth", "level", "rank", "win")
        .show(10, truncate=False))

    spark.table("lakehouse.raw.pg__game_modes").show(50, truncate=False)
    spark.table("lakehouse.raw.pg__lobby_types").show(50, truncate=False)

+----------+----------+--------+---------+----------+-------------+-----------+-------------+----------+-------+
|match_id  |start_time|duration|game_mode|lobby_type|human_players|radiant_win|radiant_score|dire_score|cluster|
+----------+----------+--------+---------+----------+-------------+-----------+-------------+----------+-------+
|9044683739|1791984795|1902    |22       |7         |10           |true       |48           |14        |154    |
|9044663515|1791984060|2632    |22       |7         |10           |true       |52           |41        |145    |
|9044661387|1791983975|2681    |22       |7         |10           |true       |38           |36        |417    |
|9044714958|1791985910|816     |23       |0         |10           |true       |23           |13        |187    |
|9044658091|1791983859|2638    |22       |7         |10           |true       |40           |47        |187    |
+----------+----------+--------+---------+----------+-------------+-----------+-------------+---

### Объёмы данных и временное окно наблюдения

Считаем количество строк, уникальных сущностей и границы окна по времени. Это определяет, насколько глубокая история доступна для каждого аккаунта.

**Выводы:**
* `pg__matches`: **317 104 строки**, все `match_id` уникальны — дублей на уровне матчей нет.
* Окно наблюдения: **14.10 12:44:45 — 15.10 06:45:05**, то есть **18.0 часов**. Это критическое ограничение: за такой период физически невозможно набрать много матчей на аккаунт.
* `pg__match_players`: **4 679 251 строка**, **467 934 уникальных матча**, **1 526 500 уникальных account_id**, **128 героев** (полный пул).
* Матчей в `match_players` больше, чем в `matches` (467 934 против 317 104) — часть строк игроков относится к матчам, отсутствующим в таблице матчей. Потребуется `INNER JOIN` для отсева «сирот».
* Соотношение 4.68M строк / 467.9K матчей ≈ 10 — соответствует полному составу, но нужна проверка полноты по каждому матчу отдельно.
* Среднее число матчей на аккаунт: 4.68M / 1.53M ≈ 3. Уже видно, что медиана будет близка к минимуму.

In [15]:
with SparkSessionContext('h1-step0-counts') as spark:
    m = spark.table("lakehouse.raw.pg__matches")
    mp = spark.table("lakehouse.raw.pg__match_players")

    m.agg(
        F.count("*").alias("rows_matches"),
        F.countDistinct("match_id").alias("uniq_match_id"),
        F.min("start_time").alias("min_start_ts"),
        F.max("start_time").alias("max_start_ts"),
        F.from_unixtime(F.min("start_time")).alias("first_match"),
        F.from_unixtime(F.max("start_time")).alias("last_match"),
    ).show(truncate=False)

    mp.agg(
        F.count("*").alias("rows_mp"),
        F.countDistinct("match_id").alias("uniq_match_id"),
        F.countDistinct("account_id").alias("uniq_account_id"),
        F.countDistinct("hero_id").alias("uniq_hero_id"),
    ).show(truncate=False)

+------------+-------------+------------+------------+-------------------+-------------------+
|rows_matches|uniq_match_id|min_start_ts|max_start_ts|first_match        |last_match         |
+------------+-------------+------------+------------+-------------------+-------------------+
|317104      |317104       |1791981885  |1792046705  |2026-10-14 12:44:45|2026-10-15 06:45:05|
+------------+-------------+------------+------------+-------------------+-------------------+



+-------+-------------+---------------+------------+
|rows_mp|uniq_match_id|uniq_account_id|uniq_hero_id|
+-------+-------------+---------------+------------+
|4679251|467934       |1526500        |128         |
+-------+-------------+---------------+------------+



In [16]:
print((1792046705  -1791981885)/3600)

18.005555555555556


### Проверка качества RAW: дубли, заглушки, пропуски, связность

Комплексный аудит перед построением признаков: естественные ключи, типы `account_id`, доля пропусков, покрытие матчей игроками, баланс сторон и распределение длительности.

**Выводы:**
* **Дублей по `(match_id, player_slot)` нет**: все 4 679 251 ключей встречаются ровно один раз. Тем не менее защита через `ROW_NUMBER` в STAGE необходима, поскольку RAW загружается в append-режиме.
* **Типы `account_id`:** нормальные — 3 095 133 строк (1 526 498 аккаунтов, avg GPM 657.9, WR 0.503); анонимы (`4294967295`) — **1 507 145 строк (32%)**; заглушка `0` — **76 973 строки**.
* Заглушка `account_id = 0` имеет **avg GPM 1167.1** — почти вдвое выше нормы (657.9) при WR 0.437. Это боты или системные записи; их включение исказило бы всю статистику. Исключаются обязательно.
* **Пропусков в игровых показателях нет** (`kills`, `deaths`, `assists`, `kda`, `gold_per_min`, `xp_per_min`, `hero_damage`, `tower_damage`, `net_worth`, `level`, `last_hits`, `hero_id`, `win` — все 0.00%).
* **`rank` заполнен только на 36.33%**: NULL в 63.67% строк (2 979 496), валидных значений 1–80 — 1 699 755. Нулевых и выходящих за диапазон нет.
* Неположительные значения редки: `gold_per_min` ≤ 0 — 0.13%, `xp_per_min` — 0.30%, `hero_damage` — 0.34%, `level` — 0.21%. Похоже на матчи с ранним выходом/афк.
* **Полнота составов:** 467 914 матчей имеют ровно 10 строк, но есть аномалии — 11 матчей с 2 строками, 5 с 9, 4 с 11.
* **Связность:** 368 матчей из `matches` не имеют ни одного игрока; **151 198 матчей** из `match_players` отсутствуют в `matches`. Обе группы отсекаются `INNER JOIN`.
* **Баланс сторон:** Radiant побеждает в 166 974 матчах против 150 130 у Dire (52.6% / 47.4%) — умеренный перекос, известный эффект карты.
* **Длительность:** p1 = 942 с, p25 = 1620, медиана 2109 с (35 мин), p75 = 2573, p99 = 3976 с. Экстремумы: min 374 с (6 мин), max 7533 с (125 мин). Обосновывает фильтр `duration BETWEEN 720 AND 5400`.

In [17]:
with SparkSessionContext('h1-04-quality') as spark:
    mp = spark.table("lakehouse.raw.pg__match_players")
    m  = spark.table("lakehouse.raw.pg__matches")

    # --- дубли по естественному ключу
    print("=== match_players: сколько строк на (match_id, player_slot) ===")
    (mp.groupBy("match_id", "player_slot").count()
       .groupBy("count").agg(F.count("*").alias("keys_cnt"))
       .orderBy("count").show(20, truncate=False))

    # --- типы account_id
    print("=== account_id ===")
    (mp.withColumn("acc_type",
            F.when(F.col("account_id").isNull(), F.lit("null"))
             .when(F.col("account_id") == 0, F.lit("zero"))
             .when(F.col("account_id") == 4294967295, F.lit("anonymous"))
             .otherwise(F.lit("normal")))
       .groupBy("acc_type")
       .agg(F.count("*").alias("rows_cnt"),
            F.countDistinct("account_id").alias("uniq_acc"),
            F.round(F.avg("gold_per_min"), 1).alias("avg_gpm"),
            F.round(F.avg(F.col("win").cast("int")), 3).alias("wr"))
       .orderBy(F.desc("rows_cnt")).show(truncate=False))

    # --- пропуски одним проходом
    print("=== nulls / non-positive, % от всех строк ===")
    cols = ["kills","deaths","assists","kda","gold_per_min","xp_per_min",
            "hero_damage","tower_damage","net_worth","level","last_hits","rank","hero_id"]
    aggs = []
    for c in cols:
        aggs.append(F.round(100 * F.avg(F.col(c).isNull().cast("int")), 2).alias(f"null_{c}"))
    for c in ["gold_per_min","xp_per_min","hero_damage","level"]:
        aggs.append(F.round(100 * F.avg((F.col(c) <= 0).cast("int")), 2).alias(f"nonpos_{c}"))
    aggs.append(F.round(100 * F.avg(F.col("win").isNull().cast("int")), 2).alias("null_win"))
    row = mp.agg(*aggs).first().asDict()
    for k, v in row.items():
        print(f"{k:26s} {v}")

    # --- ранг отдельно
    print("=== rank ===")
    (mp.withColumn("rank_state",
            F.when(F.col("rank").isNull(), F.lit("null"))
             .when(F.col("rank") == 0, F.lit("zero"))
             .when(F.col("rank").between(1, 80), F.lit("valid"))
             .otherwise(F.lit("out_of_range")))
       .groupBy("rank_state").count().orderBy(F.desc("count")).show(truncate=False))

    # --- покрытие матчей
    print("=== связь matches <-> match_players ===")
    per_match = mp.groupBy("match_id").agg(
        F.count("*").alias("rows_cnt"),
        F.countDistinct("player_slot").alias("slots_cnt"),
        F.sum(F.col("is_radiant").cast("int")).alias("radiant_cnt"))

    per_match.groupBy("rows_cnt").count().orderBy("rows_cnt").show(15, truncate=False)
    print("matches без игроков:", m.join(per_match, "match_id", "left_anti").count())
    print("игроки без матча :", mp.select("match_id").distinct().join(m, "match_id", "left_anti").count())

    # --- баланс сторон и счёт
    print("=== radiant_win / длительность ===")
    (m.groupBy("radiant_win").count().show(truncate=False))
    (m.selectExpr("percentile_approx(duration, array(0.01,0.25,0.5,0.75,0.99)) as dur_q",
                  "min(duration) as dmin", "max(duration) as dmax").show(truncate=False))

=== match_players: сколько строк на (match_id, player_slot) ===


+-----+--------+
|count|keys_cnt|
+-----+--------+
|1    |4679251 |
+-----+--------+

=== account_id ===


+---------+--------+--------+-------+-----+
|acc_type |rows_cnt|uniq_acc|avg_gpm|wr   |
+---------+--------+--------+-------+-----+
|normal   |3095133 |1526498 |657.9  |0.503|
|anonymous|1507145 |1       |643.0  |0.498|
|zero     |76973   |1       |1167.1 |0.437|
+---------+--------+--------+-------+-----+

=== nulls / non-positive, % от всех строк ===


26/11/13 06:11:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

null_kills                 0.0
null_deaths                0.0
null_assists               0.0
null_kda                   0.0
null_gold_per_min          0.0
null_xp_per_min            0.0
null_hero_damage           0.0
null_tower_damage          0.0
null_net_worth             0.0
null_level                 0.0
null_last_hits             0.0
null_rank                  63.67
null_hero_id               0.0
nonpos_gold_per_min        0.13
nonpos_xp_per_min          0.3
nonpos_hero_damage         0.34
nonpos_level               0.21
null_win                   0.0
=== rank ===


+----------+-------+
|rank_state|count  |
+----------+-------+
|null      |2979496|
|valid     |1699755|
+----------+-------+

=== связь matches <-> match_players ===


... (2 outputs left out)

игроки без матча : 151198
=== radiant_win / длительность ===


+-----------+------+
|radiant_win|count |
+-----------+------+
|true       |166974|
|false      |150130|
+-----------+------+



+-----------------------------+----+----+
|dur_q                        |dmin|dmax|
+-----------------------------+----+----+
|[942, 1620, 2109, 2573, 3976]|374 |7533|
+-----------------------------+----+----+



26/11/13 06:48:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-a53b83e2-f79a-41e2-8f51-68fd4522e961/pyspark-3dff0a8a-c810-43fc-9cca-1d8615407521. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-a53b83e2-f79a-41e2-8f51-68fd4522e961/pyspark-3dff0a8a-c810-43fc-9cca-1d8615407521
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfu

### Baseline-витрина признаков: агрегация по аккаунту и перенос в Pandas

Строим первую версию профиля игрока на сегменте Ranked All Draft: одна строка = один `account_id`, сырые средние показатели. Переносим результат в Pandas для дальнейшего анализа распределений.

**Выводы:**
* Получено **143 229 профилей × 17 признаков**. Размер отличается от предыдущего прогона (143 208) — RAW догружается в append-режиме, между запусками добавились новые матчи. Дедупликация в STAGE обеспечит идемпотентность результата.
* Фильтры применены на этапе агрегации: исключены анонимы и заглушки, оставлен только `game_mode = 22 AND lobby_type = 7`, отсечены матчи с `duration <= 0`, порог `HAVING COUNT(DISTINCT match_id) >= 3`.
* В первых строках у большинства профилей `matches_count = 3` — минимально допустимое значение. `win_rate` принимает только дискретные значения 0 / 33.33 / 66.67 / 100.
* `short_match_share = 0.000` у всех просмотренных профилей — признак неинформативен в этом сегменте.
* Часть колонок приходит в Pandas как `Decimal` (`avg_hero_damage_per_min`, `avg_tower_damage_per_min`, `win_rate`, `median_duration_min`, `short_match_share`) — следствие SQL-деления. Перед подачей в модель потребуется приведение к `float`.
* Признаки пока сырые: без учёта героя, роли и контекста матча. Это отправная точка для последующей нормировки.

In [1]:
from lakehouse_spark import SparkSessionContext

with SparkSessionContext('ml-player-features-ranked') as spark:

    # ========================================================
    # 2. Получение данных и построение признаков
    # ========================================================

    ml_player_features_ranked = spark.sql("""

        SELECT
            -- Идентификатор игрока
            p.account_id,

            -- 1. Активность игрока
            COUNT(DISTINCT p.match_id) AS matches_count,

            -- 2. Эффективность игрока
            ROUND(AVG(p.kda), 2) AS avg_kda,
            ROUND(AVG(p.gold_per_min), 2) AS avg_gpm,
            ROUND(AVG(p.xp_per_min), 2) AS avg_xpm,

            ROUND(AVG(p.kills), 2) AS avg_kills,
            ROUND(AVG(p.deaths), 2) AS avg_deaths,
            ROUND(AVG(p.assists), 2) AS avg_assists,
            ROUND(AVG(p.level), 2) AS avg_level,

            -- 3. Урон
            ROUND(AVG(p.hero_damage), 2) AS avg_hero_damage,
            ROUND(AVG(p.tower_damage), 2) AS avg_tower_damage,

            -- Урон с учётом продолжительности матча
            ROUND(
                AVG(p.hero_damage / (m.duration / 60.0)),
                2
            ) AS avg_hero_damage_per_min,

            ROUND(
                AVG(p.tower_damage / (m.duration / 60.0)),
                2
            ) AS avg_tower_damage_per_min,

            -- 4. Результативность
            ROUND(
                100.0 * AVG(
                    CASE
                        WHEN p.win = true THEN 1.0
                        ELSE 0.0
                    END
                ),
                2
            ) AS win_rate,

            -- 5. Продолжительность матчей
            ROUND(
                AVG(m.duration) / 60.0,
                2
            ) AS avg_duration_min,

            ROUND(
                percentile_approx(m.duration, 0.5) / 60.0,
                2
            ) AS median_duration_min,

            -- Доля матчей короче 7 минут
            ROUND(
                AVG(
                    CASE
                        WHEN m.duration < 420 THEN 1.0
                        ELSE 0.0
                    END
                ),
                3
            ) AS short_match_share

        FROM lakehouse.raw.pg__match_players p

        INNER JOIN lakehouse.raw.pg__matches m
            ON p.match_id = m.match_id

        WHERE
            -- Исключаем неизвестные идентификаторы игроков
            p.account_id IS NOT NULL
            AND p.account_id NOT IN (0, 4294967295)

            -- Оставляем только All Draft Ranked
            AND m.game_mode = 22
            AND m.lobby_type = 7

            -- Исключаем матчи с некорректной длительностью
            AND m.duration > 0

        GROUP BY
            p.account_id

        -- Минимум три уникальных матча на игрока
        HAVING COUNT(DISTINCT p.match_id) >= 3

    """)

    ml_player_features_ranked_pd = (
        ml_player_features_ranked.toPandas()
    )

    print("\nDataFrame для Python / Pandas:")

    print(
        "Размер:",
        ml_player_features_ranked_pd.shape
    )

    display(
        ml_player_features_ranked_pd.head()
    )


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 08:41:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/13 08:41:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                


DataFrame для Python / Pandas:
Размер: (143229, 17)


,account_id,matches_count,avg_kda,avg_gpm,avg_xpm,avg_kills,avg_deaths,avg_assists,avg_level,avg_hero_damage,avg_tower_damage,avg_hero_damage_per_min,avg_tower_damage_per_min,win_rate,avg_duration_min,median_duration_min,short_match_share
0,13000,3,4.03,385.67,596.67,3.33,8.33,25.67,21.33,14797.67,783.67,340.82,18.04,100.00,43.73,42.73,0.000
1,32113,3,2.31,461.33,619.00,6.67,8.33,11.67,20.67,25160.33,1299.00,600.43,30.72,0.00,40.83,43.18,0.000
2,34147,5,2.90,342.80,643.80,8.60,8.80,17.00,22.40,22407.00,607.20,477.53,12.88,60.00,46.27,46.37,0.000
3,44365,3,1.06,443.33,537.00,3.00,10.00,8.33,20.00,14190.33,4565.33,317.30,96.59,33.33,42.07,46.90,0.000
4,62015,3,4.09,329.00,575.33,7.00,8.33,23.33,20.67,15380.67,1223.67,361.95,26.68,66.67,42.56,44.43,0.000


### Аудит ключей и полноты составов в целевом сегменте

Уточняем проверки на актуальном снапшоте RAW и фокусируемся на главном вопросе: сколько именованных игроков остаётся в ranked-матче после исключения анонимов. Это определяет, можно ли строить корректный внутриматчевой контекст.

**Выводы:**
* **Дублей нет ни по слоту, ни по аккаунту:** 4 700 431 ключ `(match_id, player_slot)` и 3 105 060 ключей `(match_id, account_id)` среди именованных — все уникальны. Один аккаунт не может встречаться дважды в одном матче.
* Объём RAW вырос относительно первого аудита (4 679 251 → 4 700 431 строка) — подтверждение append-загрузки парсером.
* **Все 174 694 ranked-матча имеют ровно 10 строк и ровно 5 игроков Radiant.** Структурных аномалий в целевом сегменте нет, дублей слотов не обнаружено (0 матчей, где `n_rows != n_slots`).
* **Ключевая находка — распределение именованных игроков в матче:** полностью открытые составы (10 из 10) — только **116 103 матча (66.5%)**. Остальные 58 591 матч (33.5%) содержат от 1 до 9 именованных игроков.
* Распределение частично закрытых составов близко к равномерному: 4 403 матча с 1 именованным, 7 461 с 2, 9 669 с 3, 10 164 с 4, 9 510 с 5 и далее по убыванию.
* **Следствие для методологии:** внутриматчевой z-score, рассчитанный по частично закрытому составу, смещён — анонимы систематически отличаются от именованных (в первом аудите avg GPM 643.0 против 657.9). Поэтому в STAGE вводится фильтр полноты `named_players = 10 AND named_radiant = 5`, оставляющий только 116 103 матча.
* Цена фильтра — потеря трети матчей, но взамен получаем сопоставимый контекст сравнения внутри каждого матча.

In [1]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('keys-audit') as spark:
    mp = spark.table("lakehouse.raw.pg__match_players")
    m  = spark.table("lakehouse.raw.pg__matches")
    ranked = m.filter((F.col("game_mode") == 22) & (F.col("lobby_type") == 7))

    # --- 1. дубли по слоту (технические)
    print("=== строк на (match_id, player_slot), весь RAW ===")
    (mp.groupBy("match_id", "player_slot").count()
       .groupBy("count").agg(F.count("*").alias("keys_cnt"))
       .orderBy("count").show(10, truncate=False))

    # --- 2. дубли по аккаунту среди именованных
    print("=== строк на (match_id, account_id), только именованные ===")
    named = mp.filter(F.col("account_id").isNotNull() &
                      ~F.col("account_id").isin(0, 4294967295))
    (named.groupBy("match_id", "account_id").count()
       .groupBy("count").agg(F.count("*").alias("keys_cnt"))
       .orderBy("count").show(10, truncate=False))

    # --- 3. полнота матчей: вот это главное
    print("=== сколько строк игроков в ranked-матче ===")
    per_match = (mp.join(ranked.select("match_id"), "match_id")
                   .groupBy("match_id")
                   .agg(F.count("*").alias("n_rows"),
                        F.countDistinct("player_slot").alias("n_slots"),
                        F.sum(F.col("is_radiant").cast("int")).alias("n_radiant")))
    (per_match.groupBy("n_rows").count().orderBy("n_rows").show(15, truncate=False))

    print("=== баланс сторон в ranked-матче ===")
    (per_match.groupBy("n_radiant").count().orderBy("n_radiant").show(15, truncate=False))

    print("=== матчи, где n_rows != n_slots (дубли слотов) ===")
    print(per_match.filter(F.col("n_rows") != F.col("n_slots")).count())

    # --- 4. сколько именованных остаётся в матче после чистки
    print("=== именованных игроков в ranked-матче ===")
    (named.join(ranked.select("match_id"), "match_id")
       .groupBy("match_id").count()
       .groupBy("count").agg(F.count("*").alias("matches_cnt"))
       .orderBy("count").show(15, truncate=False))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 09:59:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== строк на (match_id, player_slot), весь RAW ===


+-----+--------+
|count|keys_cnt|
+-----+--------+
|1    |4700431 |
+-----+--------+

=== строк на (match_id, account_id), только именованные ===


+-----+--------+
|count|keys_cnt|
+-----+--------+
|1    |3105060 |
+-----+--------+

=== сколько строк игроков в ranked-матче ===


+------+------+
|n_rows|count |
+------+------+
|10    |174694|
+------+------+

=== баланс сторон в ranked-матче ===


+---------+------+
|n_radiant|count |
+---------+------+
|5        |174694|
+---------+------+

=== матчи, где n_rows != n_slots (дубли слотов) ===


0
=== именованных игроков в ranked-матче ===


+-----+-----------+
|count|matches_cnt|
+-----+-----------+
|1    |4403       |
|2    |7461       |
|3    |9669       |
|4    |10164      |
|5    |9510       |
|6    |7479       |
|7    |4831       |
|8    |2521       |
|9    |935        |
|10   |116103     |
+-----+-----------+



### Заполненность ранга и влияние полноты состава на статистику

Проверяем два вопроса: насколько заполнен `rank` в целевом сегменте и искажает ли частичная закрытость профилей игровые показатели. Это определяет, нужен ли фильтр полноты состава в STAGE.

**Выводы:**
* В сегменте Ranked All Draft **1 767 230 строк**, из них `rank` заполнен у **1 157 188 (65.5%)**, NULL — 610 042 (34.5%). Нулевых и выходящих за диапазон 1–80 значений нет.
* **Ключевая находка:** ранг заполнен **только в полностью открытых матчах**. При `named_cnt < 10` средний ранг = NULL во всех группах; при `named_cnt = 10` заполнено 9.87 из 10 слотов, средняя медаль 49.7 (Legend).
* Это означает: приватность профиля — свойство игрока, а не матча. Если хотя бы один участник скрыл профиль, ранг недоступен ни у кого в этом матче (особенность выгрузки API).
* Полностью открытых матчей — **117 545 (66.5%)**, частично закрытых — 59 178 (33.5%). Распределение по числу именованных близко к равномерному от 1 до 9.
* **Систематическое смещение подтверждено:** при `named_cnt = 1..9` средний GPM именованных игроков составляет 487–497, XPM — 702–725. При `named_cnt = 10` — GPM 476.2, XPM 683.7. Разница **+3–4% по GPM и +3–6% по XPM**.
* Причина: в частично закрытых матчах именованные игроки остаются «видимой» частью выборки, а анонимы систематически слабее. Внутриматчевой z-score, посчитанный по такому составу, будет смещён.
* Длительность матчей также различается: 42.0–42.3 мин при неполном составе против 40.4 мин при полном.
* **По регионам:** доля полностью открытых матчей варьируется от 0.573 (кластер 152) до 0.771 (кластер 251). Кластеры 151–153 заметно «закрытее» остальных — вероятно, региональная особенность настроек приватности.
* **Решение:** ввести в STAGE фильтр `named_players = 10 AND named_radiant = 5`. Цена — потеря трети матчей, взамен получаем корректный контекст сравнения и доступность ранга.

In [1]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('rank-and-coverage') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7))
    mp = spark.table("lakehouse.raw.pg__match_players")

    j = mp.join(m.select("match_id", "duration", "cluster", "start_time"), "match_id")

    # --- заполненность rank
    print("=== rank: заполненность ===")
    (j.agg(F.count("*").alias("rows"),
           F.sum(F.col("rank").isNull().cast("int")).alias("rank_null"),
           F.sum((F.col("rank") == 0).cast("int")).alias("rank_zero"),
           F.sum(F.col("rank").between(1, 80).cast("int")).alias("rank_valid"))
        .show(truncate=False))

    # --- покрытие матча
    cov = j.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"),
        F.avg(F.when(F.col("rank").between(1, 80), F.col("rank"))).alias("avg_rank"),
        F.sum(F.col("rank").between(1, 80).cast("int")).alias("rank_cnt"),
        F.first("duration").alias("duration"),
        F.first("cluster").alias("cluster"))

    print("=== покрытие vs медиана статы и ранг ===")
    (cov.groupBy("named_cnt")
        .agg(F.count("*").alias("matches"),
             F.round(F.avg("duration") / 60.0, 1).alias("avg_min"),
             F.round(F.avg("avg_rank"), 1).alias("avg_medal_num"),
             F.round(F.avg("rank_cnt"), 2).alias("avg_rank_filled"))
        .orderBy("named_cnt").show(15, truncate=False))

    print("=== покрытие по регионам (топ-15 кластеров) ===")
    (cov.groupBy("cluster")
        .agg(F.count("*").alias("matches"),
             F.round(F.avg("named_cnt"), 2).alias("avg_named"),
             F.round(F.avg(F.when(F.col("named_cnt") == 10, 1).otherwise(0)), 3).alias("share_full_open"))
        .orderBy(F.desc("matches")).show(15, truncate=False))

    print("=== стата игроков в зависимости от покрытия матча ===")
    (j.join(cov.select("match_id", "named_cnt"), "match_id")
        .filter(F.col("account_id").isNotNull() & ~F.col("account_id").isin(0, 4294967295))
        .groupBy("named_cnt")
        .agg(F.count("*").alias("rows"),
             F.round(F.avg("gold_per_min"), 1).alias("gpm"),
             F.round(F.avg("xp_per_min"), 1).alias("xpm"),
             F.round(F.avg("kda"), 2).alias("kda"),
             F.round(F.avg("denies"), 1).alias("denies"),
             F.round(F.avg(F.col("win").cast("int")), 3).alias("wr"))
        .orderBy("named_cnt").show(15, truncate=False))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 04:14:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== rank: заполненность ===


+-------+---------+---------+----------+
|rows   |rank_null|rank_zero|rank_valid|
+-------+---------+---------+----------+
|1767230|610042   |0        |1157188   |
+-------+---------+---------+----------+

=== покрытие vs медиана статы и ранг ===


+---------+-------+-------+-------------+---------------+
|named_cnt|matches|avg_min|avg_medal_num|avg_rank_filled|
+---------+-------+-------+-------------+---------------+
|0        |1630   |42.0   |NULL         |NULL           |
|1        |4452   |42.1   |NULL         |NULL           |
|2        |7562   |42.3   |NULL         |NULL           |
|3        |9752   |42.2   |NULL         |NULL           |
|4        |10270  |41.9   |NULL         |NULL           |
|5        |9602   |41.9   |NULL         |NULL           |
|6        |7555   |41.7   |NULL         |NULL           |
|7        |4873   |41.5   |NULL         |NULL           |
|8        |2541   |41.5   |NULL         |NULL           |
|9        |941    |40.9   |NULL         |NULL           |
|10       |117545 |40.4   |49.7         |9.87           |
+---------+-------+-------+-------------+---------------+

=== покрытие по регионам (топ-15 кластеров) ===


+-------+-------+---------+---------------+
|cluster|matches|avg_named|share_full_open|
+-------+-------+---------+---------------+
|185    |11567  |8.15     |0.664          |
|183    |10367  |8.11     |0.655          |
|182    |10155  |8.1      |0.655          |
|184    |9887   |8.15     |0.662          |
|181    |9716   |8.13     |0.66           |
|152    |8624   |7.27     |0.573          |
|151    |8474   |7.33     |0.584          |
|381    |8369   |8.2      |0.766          |
|251    |8196   |8.24     |0.771          |
|187    |7659   |8.14     |0.662          |
|153    |7255   |7.35     |0.588          |
|189    |6582   |8.01     |0.679          |
|273    |6476   |8.24     |0.686          |
|186    |6406   |8.11     |0.657          |
|272    |6007   |8.28     |0.691          |
+-------+-------+---------+---------------+
only showing top 15 rows
=== стата игроков в зависимости от покрытия матча ===


+---------+-------+-----+-----+----+------+-----+
|named_cnt|rows   |gpm  |xpm  |kda |denies|wr   |
+---------+-------+-----+-----+----+------+-----+
|1        |4452   |487.4|722.1|4.39|7.0   |0.526|
|2        |15124  |491.3|725.3|4.52|7.3   |0.515|
|3        |29256  |491.5|719.5|4.39|7.4   |0.514|
|4        |41080  |494.4|719.0|4.39|7.6   |0.508|
|5        |48010  |495.3|715.1|4.35|7.5   |0.505|
|6        |45330  |496.1|713.0|4.33|7.6   |0.506|
|7        |34111  |496.9|709.8|4.33|7.6   |0.502|
|8        |20328  |497.0|709.7|4.36|7.6   |0.501|
|9        |8469   |495.1|701.7|4.41|7.6   |0.497|
|10       |1175450|476.2|683.7|4.31|7.0   |0.5  |
+---------+-------+-----+-----+----+------+-----+



### Внутриматчевая нормировка: проверка z-score и величины разброса

Строим первый уровень нормировки — z-score внутри матча. Проверяем корректность расчёта и оцениваем, насколько велик естественный разброс показателей между игроками одного матча.

**Выводы:**
* **Sanity check пройден:** `mean(z_dmg) = 0.0`, `mean(z_gpm) = 0.0`, `std(z_dmg) = 0.9487` (близко к 1, отклонение из-за `stddev_samp` vs `stddev`), пропусков нет (`null_z = 0`). Доля топ-1 по урону = 0.1 — ровно 1 из 10, как и должно быть.
* Обработано **176 662 матча** после фильтра `human_players = 10 AND duration >= 720`.
* **Разброс урона внутри матча велик:** коэффициент вариации `std/mean` имеет медиану **0.43**, p10 = 0.31, p90 = 0.58. То есть типичное стандартное отклонение урона составляет почти половину среднего — сравнивать абсолютные значения между матчами бессмысленно.
* **Разброс GPM:** медианное std = **153.6** при среднем GPM около 480. p10 = 119.7, p90 = 191.0. Керри и саппорт в одном матче различаются на 300+ GPM — это норма, а не аномалия.
* **Denies содержат сигнал:** медиана 5, p75 = 10, p90 = 16, p99 = 29. Доля нулевых denies — всего **8.4%**, то есть большинство игроков денают хотя бы иногда. Признак не вырожден.
* Denies в минуту: медиана 0.13, p90 = 0.40, p99 = 0.77. Разброс на порядок — хороший потенциал для дискриминации механического скилла.
* Максимальные denies в матче: медиана 18, p90 = 29, p99 = 42. Всегда есть игрок с заметным числом добиваний.
* **Вывод:** внутриматчевой z-score корректно центрирует показатели и убирает влияние длительности, уровня лобби и патча. Но он не убирает влияние роли — для этого потребуется отдельная нормировка.

In [2]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext('within-match-spread') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")

    base = mp.join(m.select("match_id", "duration"), "match_id")
    w = Window.partitionBy("match_id")

    ctx = (base
        .withColumn("z_dmg", (F.col("hero_damage") - F.avg("hero_damage").over(w))
                            / F.nullif(F.stddev_samp("hero_damage").over(w), F.lit(0)))
        .withColumn("z_gpm", (F.col("gold_per_min") - F.avg("gold_per_min").over(w))
                            / F.nullif(F.stddev_samp("gold_per_min").over(w), F.lit(0)))
        .withColumn("rank_dmg", F.rank().over(w.orderBy(F.desc("hero_damage"))))
        .withColumn("farm_role", F.rank().over(
            Window.partitionBy("match_id", "is_radiant").orderBy(F.desc("net_worth")))))

    print("=== sanity check: z должен быть ~0, top1 ~0.1 ===")
    (ctx.agg(F.round(F.avg("z_dmg"), 4).alias("mean_z_dmg"),
             F.round(F.stddev("z_dmg"), 4).alias("std_z_dmg"),
             F.round(F.avg("z_gpm"), 4).alias("mean_z_gpm"),
             F.round(F.avg(F.when(F.col("rank_dmg") == 1, 1.0).otherwise(0.0)), 4).alias("share_top1"),
             F.sum(F.col("z_dmg").isNull().cast("int")).alias("null_z"))
        .show(truncate=False))

    print("=== насколько велик разброс внутри матча ===")
    (base.groupBy("match_id")
        .agg(F.stddev_samp("hero_damage").alias("s_dmg"),
             F.avg("hero_damage").alias("m_dmg"),
             F.stddev_samp("gold_per_min").alias("s_gpm"),
             F.max("denies").alias("max_denies"),
             F.min("denies").alias("min_denies"))
        .selectExpr("count(*) as matches",
                    "percentile_approx(s_dmg/m_dmg, array(0.1,0.5,0.9)) as cv_dmg_q",
                    "percentile_approx(s_gpm, array(0.1,0.5,0.9)) as std_gpm_q",
                    "percentile_approx(max_denies, array(0.5,0.9,0.99)) as max_denies_q")
        .show(truncate=False))

    print("=== denies: есть ли вообще сигнал ===")
    (base.withColumn("denies_per_min", F.col("denies") / (F.col("duration") / 60.0))
        .selectExpr("percentile_approx(denies, array(0.1,0.25,0.5,0.75,0.9,0.99)) as denies_q",
                    "percentile_approx(denies_per_min, array(0.5,0.9,0.99)) as dpm_q",
                    "avg(case when denies = 0 then 1.0 else 0.0 end) as share_zero_denies")
        .show(truncate=False))

=== sanity check: z должен быть ~0, top1 ~0.1 ===


+----------+---------+----------+----------+------+
|mean_z_dmg|std_z_dmg|mean_z_gpm|share_top1|null_z|
+----------+---------+----------+----------+------+
|0.0       |0.9487   |0.0       |0.1       |0     |
+----------+---------+----------+----------+------+

=== насколько велик разброс внутри матча ===


+-------+------------------------------------------------------------+------------------------------------------------------------+------------+
|matches|cv_dmg_q                                                    |std_gpm_q                                                   |max_denies_q|
+-------+------------------------------------------------------------+------------------------------------------------------------+------------+
|176662 |[0.3084036127581639, 0.4295529023667192, 0.5846575245197676]|[119.65714911083806, 153.63000285679155, 190.99159376498454]|[18, 29, 42]|
+-------+------------------------------------------------------------+------------------------------------------------------------+------------+

=== denies: есть ли вообще сигнал ===


+---------------------+--------------------------------------------------------------+-----------------+
|denies_q             |dpm_q                                                         |share_zero_denies|
+---------------------+--------------------------------------------------------------+-----------------+
|[1, 2, 5, 10, 16, 29]|[0.12987012987012986, 0.40358744394618834, 0.7729112992270888]|0.08388          |
+---------------------+--------------------------------------------------------------+-----------------+



### Однородность лобби по рангу и зависимость статистики от медали

Проверяем, можно ли использовать ранг как контекст силы игрока: насколько однородны лобби, растёт ли статистика с медалью, и сохраняется ли этот рост внутри одной позиции.

**Выводы:**
* После фильтра полноты состава осталось **117 501 матч / 1 175 010 строк**.
* **Лобби умеренно однородны:** медианное std ранга внутри матча = **3.95** пункта (меньше половины звезды), p90 = 9.57, p99 = 13.0. Средний ранг матча: p10 = 29.9, медиана 50.1, p90 = 70.3 — покрыт весь диапазон от Crusader до Immortal.
* **Статистика монотонно растёт с медалью** (все роли): GPM с 418.6 (medal 1) до 506.1 (medal 7), denies/min с 0.117 до 0.197, last_hits со 152.7 до 213.7, WR с 0.461 до 0.514. Градиент подтверждает: ранг отражает реальный уровень игры.
* **Но эффект роли на порядок сильнее эффекта медали.** Внутри `farm_role = 1` (керри) GPM растёт с 587.5 до 689.2 (+17%), а между ролями 1 и 5 разница 655.9 → 324.6 (**−50%**). Роль доминирует.
* **z_dmg внутри роли почти не зависит от медали:** для керри значения 0.737–0.787 по всем медалям, для саппорта −0.748…−0.766. Это означает, что внутриматчевой z-score уже снял основной эффект силы лобби — ранг добавляет мало информации сверх роли.
* Исключение — `farm_role = 2`: z_dmg растёт с 0.187 до 0.480. На второй позиции сильные игроки действительно выделяются заметнее.
* **Кандидаты «низкий ранг против сильного лобби»** (gap < −8, z_dmg > 0.8, n ≥ 3): найдены аккаунты с разрывом до −34.8 пункта при среднем z_dmg 1.25 и WR 0.71. Формально это похоже на смурфов.

In [3]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext('rank-curve') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")

    base = mp.join(m.select("match_id", "duration", "cluster"), "match_id")

    # только полностью открытые матчи
    cov = base.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"))
    full_ids = cov.filter(F.col("named_cnt") == 10).select("match_id")
    clean = base.join(full_ids, "match_id")

    print("чистых матчей:", full_ids.count(), "строк:", clean.count())

    w = Window.partitionBy("match_id")
    ctx = (clean
        .withColumn("match_avg_rank", F.avg("rank").over(w))
        .withColumn("match_std_rank", F.stddev_samp("rank").over(w))
        .withColumn("rank_gap", F.col("rank") - F.col("match_avg_rank"))
        .withColumn("z_dmg", (F.col("hero_damage") - F.avg("hero_damage").over(w))
                    / F.nullif(F.stddev_samp("hero_damage").over(w), F.lit(0)))
        .withColumn("z_gpm", (F.col("gold_per_min") - F.avg("gold_per_min").over(w))
                    / F.nullif(F.stddev_samp("gold_per_min").over(w), F.lit(0)))
        .withColumn("farm_role", F.rank().over(
            Window.partitionBy("match_id", "is_radiant").orderBy(F.desc("net_worth"))))
        .withColumn("dpm", F.col("denies") / (F.col("duration") / 60.0)))

    # --- насколько однородны лобби по рангу (важно!)
    print("=== разброс рангов внутри матча ===")
    (ctx.select("match_id", "match_avg_rank", "match_std_rank").distinct()
        .selectExpr("count(*) as matches",
                    "percentile_approx(match_std_rank, array(0.1,0.5,0.9,0.99)) as std_rank_q",
                    "percentile_approx(match_avg_rank, array(0.1,0.5,0.9)) as avg_rank_q")
        .show(truncate=False))

    # --- кривая: стата по медали
    print("=== стата по медали (все роли) ===")
    (ctx.filter(F.col("rank").between(1, 80))
        .withColumn("medal", F.floor((F.col("rank") - 1) / 10))
        .groupBy("medal")
        .agg(F.count("*").alias("rows"),
             F.round(F.avg("gold_per_min"), 1).alias("gpm"),
             F.round(F.avg("xp_per_min"), 1).alias("xpm"),
             F.round(F.avg("kda"), 2).alias("kda"),
             F.round(F.avg("dpm"), 3).alias("denies_pm"),
             F.round(F.avg("z_dmg"), 3).alias("z_dmg"),
             F.round(F.avg("last_hits"), 1).alias("lh"),
             F.round(F.avg(F.col("win").cast("int")), 3).alias("wr"))
        .orderBy("medal").show(truncate=False))

    # --- то же, но внутри роли: убираем эффект позиции
    print("=== стата по медали внутри farm_role ===")
    (ctx.filter(F.col("rank").between(1, 80))
        .withColumn("medal", F.floor((F.col("rank") - 1) / 10))
        .groupBy("farm_role", "medal")
        .agg(F.count("*").alias("rows"),
             F.round(F.avg("gold_per_min"), 1).alias("gpm"),
             F.round(F.avg("dpm"), 3).alias("denies_pm"),
             F.round(F.avg("z_dmg"), 3).alias("z_dmg"))
        .orderBy("farm_role", "medal").show(50, truncate=False))

    # --- главное: низкий ранг против сильного лобби
    print("=== кандидаты: ранг ниже лобби, но доминирует ===")
    (ctx.filter(F.col("rank").between(1, 80))
        .groupBy("account_id")
        .agg(F.count("*").alias("n"),
             F.round(F.avg("rank"), 1).alias("my_rank"),
             F.round(F.avg("match_avg_rank"), 1).alias("lobby_rank"),
             F.round(F.avg("rank_gap"), 1).alias("gap"),
             F.round(F.avg("z_dmg"), 2).alias("z_dmg"),
             F.round(F.avg("z_gpm"), 2).alias("z_gpm"),
             F.round(F.avg("dpm"), 3).alias("denies_pm"),
             F.round(F.avg(F.col("win").cast("int")), 2).alias("wr"))
        .filter((F.col("n") >= 3) & (F.col("gap") < -8) & (F.col("z_dmg") > 0.8))
        .orderBy(F.asc("gap")).show(30, truncate=False))

чистых матчей: 117501 строк: 1175010
=== разброс рангов внутри матча ===


+-------+-------------------------------------------------------------------------------+---------------------------------------------+
|matches|std_rank_q                                                                     |avg_rank_q                                   |
+-------+-------------------------------------------------------------------------------+---------------------------------------------+
|117501 |[1.0749676997731394, 3.9510898637098992, 9.573690801125528, 13.016655996581202]|[29.88888888888889, 50.111111111111114, 70.3]|
+-------+-------------------------------------------------------------------------------+---------------------------------------------+

=== стата по медали (все роли) ===


+-----+------+-----+-----+----+---------+------+-----+-----+
|medal|rows  |gpm  |xpm  |kda |denies_pm|z_dmg |lh   |wr   |
+-----+------+-----+-----+----+---------+------+-----+-----+
|1    |25809 |418.6|635.3|3.71|0.117    |-0.155|152.7|0.461|
|2    |82361 |438.2|657.6|3.95|0.143    |-0.101|166.3|0.486|
|3    |151582|453.0|670.0|4.05|0.161    |-0.071|179.0|0.49 |
|4    |250016|466.7|679.7|4.17|0.176    |-0.035|190.2|0.493|
|5    |308197|484.0|692.5|4.37|0.188    |0.024 |202.5|0.504|
|6    |178881|500.9|705.3|4.59|0.197    |0.081 |214.5|0.513|
|7    |159923|506.1|695.1|4.65|0.197    |0.072 |213.7|0.514|
+-----+------+-----+-----+----+---------+------+-----+-----+

=== стата по медали внутри farm_role ===


+---------+-----+-----+-----+---------+------+
|farm_role|medal|rows |gpm  |denies_pm|z_dmg |
+---------+-----+-----+-----+---------+------+
|1        |1    |4006 |587.5|0.194    |0.74  |
|1        |2    |13843|609.9|0.233    |0.737 |
|1        |3    |27217|629.5|0.256    |0.737 |
|1        |4    |47726|644.7|0.278    |0.748 |
|1        |5    |63613|660.8|0.289    |0.773 |
|1        |6    |40289|677.0|0.296    |0.787 |
|1        |7    |35174|689.2|0.296    |0.752 |
|2        |1    |4639 |487.6|0.154    |0.187 |
|2        |2    |15363|512.3|0.192    |0.273 |
|2        |3    |29042|527.4|0.216    |0.308 |
|2        |4    |48697|541.4|0.235    |0.358 |
|2        |5    |62997|556.0|0.249    |0.41  |
|2        |6    |37699|568.9|0.257    |0.461 |
|2        |7    |33554|580.0|0.259    |0.48  |
|3        |1    |5205 |421.6|0.12     |-0.13 |
|3        |2    |16682|441.8|0.152    |-0.07 |
|3        |3    |30648|453.6|0.175    |-0.04 |
|3        |4    |50480|465.0|0.191    |-0.005|
|3        |5 

+----------+---+-------+----------+-----+-----+-----+---------+----+
|account_id|n  |my_rank|lobby_rank|gap  |z_dmg|z_gpm|denies_pm|wr  |
+----------+---+-------+----------+-----+-----+-----+---------+----+
|1769252246|7  |12.0   |46.8      |-34.8|1.25 |1.31 |0.511    |0.71|
|1281810567|3  |35.0   |61.9      |-26.9|1.11 |1.85 |0.274    |1.0 |
|775592157 |3  |15.0   |41.8      |-26.8|1.24 |1.69 |0.55     |0.67|
|328633506 |4  |21.0   |47.0      |-26.0|2.08 |2.23 |0.576    |1.0 |
|800449166 |4  |13.0   |36.8      |-23.8|0.99 |1.51 |0.275    |0.5 |
|170493807 |4  |22.0   |45.6      |-23.6|1.06 |1.52 |0.299    |0.75|
|144205161 |3  |32.0   |55.4      |-23.4|1.52 |1.13 |0.179    |0.33|
|1855496039|3  |24.0   |47.1      |-23.1|1.15 |0.82 |0.191    |1.0 |
|445107437 |3  |25.0   |47.3      |-22.3|1.31 |1.25 |0.212    |1.0 |
|1038231564|3  |35.0   |57.1      |-22.1|1.23 |0.98 |0.149    |1.0 |
|688635046 |3  |23.0   |44.8      |-21.8|2.24 |1.92 |0.881    |1.0 |
|771032096 |3  |25.0   |46.7      

### Стабильность ранга внутри аккаунта

Смотрим, насколько меняется ранг одного аккаунта между его матчами. Если ранг фиксирован на момент игры, разброс должен быть близок к нулю при окне 18 часов.

**Выводы:**
* Проанализировано **103 356 аккаунтов** с ≥3 матчами и валидным рангом.
* **У 63% аккаунтов ранг постоянен** (`share_constant = 0.62984`), медианный размах = 0. Это ожидаемо для окна 18 часов.
* Но у остальных разброс значителен: p90 размаха = **6 пунктов**, p99 = **10 пунктов**. Медианное std = 0.0, p90 = 3.20, p99 = 5.20.
* **Обнаружены аномальные скачки:** аккаунт 799781180 — ранг с 13 до 52 (**+39 пунктов, почти 4 медали**) за 5 матчей; 232832725 — с 32 до 71 (+39); 1438384420 — с 15 до 45 (+30); 833431650 — с 13 до 41 (+28).
* Подняться на 4 медали за 5 матчей физически невозможно: одна победа даёт ~25 MMR, а переход между медалями требует ~150–200 MMR. Даже при 100% винрейте это десятки игр.
* Часть аккаунтов показывает **обратное движение**: 196216522 (`rank_first = 52`, `rank_last = 32`), 170754340 (72 → 52). Падение на 20 пунктов за 7 матчей также нереалистично.
* **Гипотеза:** ранг привязан не к дате матча, а к дате загрузки записи парсером. Требуется прямая проверка через сопоставление `start_time` и `sys_insert_stamp`.

In [4]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('rank-stability') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players").filter(
        F.col("account_id").isNotNull() & ~F.col("account_id").isin(0, 4294967295))

    j = mp.join(m.select("match_id", "start_time"), "match_id").filter(F.col("rank").between(1, 80))

    acc = j.groupBy("account_id").agg(
        F.count("*").alias("n"),
        F.min("rank").alias("rank_min"),
        F.max("rank").alias("rank_max"),
        F.stddev_samp("rank").alias("rank_std"),
        F.first("rank").alias("rank_first"),
        F.last("rank").alias("rank_last"))

    print("=== разброс ранга внутри аккаунта ===")
    (acc.filter(F.col("n") >= 3)
        .withColumn("rank_range", F.col("rank_max") - F.col("rank_min"))
        .selectExpr("count(*) as accounts",
                    "percentile_approx(rank_range, array(0.5,0.9,0.99)) as range_q",
                    "percentile_approx(rank_std, array(0.5,0.9,0.99)) as std_q",
                    "avg(case when rank_range = 0 then 1.0 else 0.0 end) as share_constant")
        .show(truncate=False))

    print("=== аккаунты с большим скачком ранга ===")
    (acc.filter((F.col("n") >= 5) & (F.col("rank_max") - F.col("rank_min") > 10))
        .orderBy(F.desc(F.col("rank_max") - F.col("rank_min")))
        .show(20, truncate=False))

=== разброс ранга внутри аккаунта ===


+--------+----------+--------------------------------------------+--------------+
|accounts|range_q   |std_q                                       |share_constant|
+--------+----------+--------------------------------------------+--------------+
|103356  |[0, 6, 10]|[0.0, 3.2015621187164243, 5.196152422706632]|0.62984       |
+--------+----------+--------------------------------------------+--------------+

=== аккаунты с большим скачком ранга ===


+----------+---+--------+--------+------------------+----------+---------+
|account_id|n  |rank_min|rank_max|rank_std          |rank_first|rank_last|
+----------+---+--------+--------+------------------+----------+---------+
|799781180 |5  |13      |52      |20.166804407243106|13        |52       |
|232832725 |6  |32      |71      |15.921683328090658|32        |71       |
|1438384420|5  |15      |45      |13.416407864998739|15        |45       |
|833431650 |6  |13      |41      |11.775681155103795|13        |41       |
|688235677 |6  |35      |61      |10.40512694140089 |35        |61       |
|834628463 |7  |21      |44      |11.222850838908132|21        |44       |
|102392167 |5  |22      |45      |10.13903348450926 |22        |45       |
|690634234 |5  |33      |55      |9.654014708917737 |33        |55       |
|685484172 |14 |21      |43      |7.397653325567151 |21        |43       |
|1770443537|5  |13      |35      |7.797435475847171 |13        |35       |
|834232361 |5  |43      |

### Прямая проверка: ранг привязан к дате игры или к дате загрузки?

Берём три аккаунта с наибольшим скачком ранга и выводим их матчи в двух сортировках — по времени матча и по времени загрузки записи. Колонка `lag_days` показывает разрыв между этими датами.

**Выводы:**
* **Гипотеза подтверждена однозначно.** У всех трёх аккаунтов ранг монотонно растёт вместе с `sys_insert_stamp`, а не со `start_time`.
* **Аккаунт 799781180:** все 5 матчей сыграны 14 октября с 13:34 до 20:11 (за 6.6 часа). Записи загружены с 19 октября по 4 ноября. Ранг: 13 (lag 4.5 дня) → 13 (lag 9.5) → 44 (lag 17.4) → 52 (lag 19.0) → 52 (lag 20.7). Рост ранга строго следует за увеличением лага загрузки.
* **Аккаунт 232832725:** 6 матчей за 10.4 часа, лаг от 4.8 до 24.7 дня. Ранг 32 → 71 при переходе лага с 4.8 на 19.5 дня, далее стабилен на 71.
* **Аккаунт 1438384420:** 5 матчей за 6.8 часа, лаг 6.0 → 21.8 дня. Ранг 15 → 45 по той же закономерности.
* **Лаг достигает 24.7 дня** — записи о матчах создавались спустя недели после самой игры. Поле `rank` в каждой записи отражает состояние профиля **на момент парсинга**, а не на момент матча.
* Обе сортировки дают идентичный порядок строк, поскольку парсер обрабатывал матчи последовательно. Но это не отменяет вывода: изменения ранга объясняются датами загрузки, а не игровым прогрессом.
* **Следствие для методологии:** поле `rank` непригодно ни как контекст силы игрока на момент матча, ни для анализа динамики («резкий рост ранга → передача аккаунта»). Гипотеза H1 отвергается, признак исключается из модели.

In [2]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('rank-vs-load-date') as spark:

    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7))

    mp = spark.table("lakehouse.raw.pg__match_players").filter(
        F.col("account_id").isNotNull() &
        ~F.col("account_id").isin(0, 4294967295) &
        F.col("rank").between(1, 80))

    j = mp.join(m.select("match_id", "start_time"), "match_id")

    # находим аккаунты, у которых ранг сильно менялся
    changing = (j.groupBy("account_id")
        .agg(F.count("*").alias("n"),
             F.min("rank").alias("rmin"),
             F.max("rank").alias("rmax"))
        .filter((F.col("n") >= 5) & ((F.col("rmax") - F.col("rmin")) >= 15))
        .orderBy(F.desc(F.col("rmax") - F.col("rmin")))
        .limit(3))

    accounts = [r["account_id"] for r in changing.collect()]
    print("Аккаунты для просмотра:", accounts)

    cols = [
        "match_id",
        F.from_unixtime("start_time").alias("played_at"),
        "sys_insert_stamp",
        F.round((F.unix_timestamp("sys_insert_stamp") - F.col("start_time")) / 86400.0, 1).alias("lag_days"),
        "rank", "win", "kda", "gold_per_min",
    ]

    for aid in accounts:
        d = j.filter(F.col("account_id") == aid)
        print(f"\n{'='*70}\nаккаунт {aid}")
        print("\n--- сортировка по дате ИГРЫ ---")
        d.select(*cols).orderBy("played_at").show(20, truncate=False)
        print("--- сортировка по дате ЗАГРУЗКИ ---")
        d.select(*cols).orderBy("sys_insert_stamp").show(20, truncate=False)

Аккаунты для просмотра: [232832725, 799781180, 1438384420]

аккаунт 232832725

--- сортировка по дате ИГРЫ ---


+----------+-------------------+--------------------------+--------+----+-----+-----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win  |kda  |gold_per_min|
+----------+-------------------+--------------------------+--------+----+-----+-----+------------+
|9044723012|2026-10-14 13:56:38|2026-10-19 08:20:23.501845|4.8     |32  |true |32.0 |721         |
|9045282403|2026-10-14 19:47:24|2026-11-03 08:46:39.120423|19.5    |71  |true |10.67|669         |
|9045386844|2026-10-14 21:25:59|2026-11-06 08:36:50.396073|22.5    |71  |false|2.0  |501         |
|9045432624|2026-10-14 22:27:03|2026-11-07 05:10:20.328852|23.3    |71  |true |16.5 |836         |
|9045456753|2026-10-14 23:04:08|2026-11-07 17:28:13.035264|23.8    |71  |true |4.0  |656         |
|9045498720|2026-10-15 00:20:04|2026-11-08 16:46:36.048579|24.7    |71  |true |3.83 |679         |
+----------+-------------------+--------------------------+--------+----+-----+-----+------------+

--- сорти

+----------+-------------------+--------------------------+--------+----+-----+-----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win  |kda  |gold_per_min|
+----------+-------------------+--------------------------+--------+----+-----+-----+------------+
|9044723012|2026-10-14 13:56:38|2026-10-19 08:20:23.501845|4.8     |32  |true |32.0 |721         |
|9045282403|2026-10-14 19:47:24|2026-11-03 08:46:39.120423|19.5    |71  |true |10.67|669         |
|9045386844|2026-10-14 21:25:59|2026-11-06 08:36:50.396073|22.5    |71  |false|2.0  |501         |
|9045432624|2026-10-14 22:27:03|2026-11-07 05:10:20.328852|23.3    |71  |true |16.5 |836         |
|9045456753|2026-10-14 23:04:08|2026-11-07 17:28:13.035264|23.8    |71  |true |4.0  |656         |
|9045498720|2026-10-15 00:20:04|2026-11-08 16:46:36.048579|24.7    |71  |true |3.83 |679         |
+----------+-------------------+--------------------------+--------+----+-----+-----+------------+


аккаунт 

+----------+-------------------+--------------------------+--------+----+----+----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win |kda |gold_per_min|
+----------+-------------------+--------------------------+--------+----+----+----+------------+
|9044686318|2026-10-14 13:34:45|2026-10-19 01:09:33.577711|4.5     |13  |true|9.5 |642         |
|9044847533|2026-10-14 15:09:05|2026-10-24 04:20:43.367684|9.5     |13  |true|10.6|906         |
|9045185347|2026-10-14 18:38:39|2026-11-01 04:30:23.239258|17.4    |44  |true|8.75|584         |
|9045256223|2026-10-14 19:28:04|2026-11-02 20:33:17.783783|19.0    |52  |true|23.0|723         |
|9045311324|2026-10-14 20:11:19|2026-11-04 13:11:56.298444|20.7    |52  |true|7.14|862         |
+----------+-------------------+--------------------------+--------+----+----+----+------------+

--- сортировка по дате ЗАГРУЗКИ ---


+----------+-------------------+--------------------------+--------+----+----+----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win |kda |gold_per_min|
+----------+-------------------+--------------------------+--------+----+----+----+------------+
|9044686318|2026-10-14 13:34:45|2026-10-19 01:09:33.577711|4.5     |13  |true|9.5 |642         |
|9044847533|2026-10-14 15:09:05|2026-10-24 04:20:43.367684|9.5     |13  |true|10.6|906         |
|9045185347|2026-10-14 18:38:39|2026-11-01 04:30:23.239258|17.4    |44  |true|8.75|584         |
|9045256223|2026-10-14 19:28:04|2026-11-02 20:33:17.783783|19.0    |52  |true|23.0|723         |
|9045311324|2026-10-14 20:11:19|2026-11-04 13:11:56.298444|20.7    |52  |true|7.14|862         |
+----------+-------------------+--------------------------+--------+----+----+----+------------+


аккаунт 1438384420

--- сортировка по дате ИГРЫ ---


+----------+-------------------+--------------------------+--------+----+-----+----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win  |kda |gold_per_min|
+----------+-------------------+--------------------------+--------+----+-----+----+------------+
|9044763059|2026-10-14 14:19:59|2026-10-20 14:17:49.057684|6.0     |15  |true |24.0|510         |
|9045210564|2026-10-14 18:55:24|2026-11-02 00:44:28.869242|18.2    |45  |true |2.67|680         |
|9045280878|2026-10-14 19:46:39|2026-11-03 04:17:24.719599|19.4    |45  |false|2.14|340         |
|9045327747|2026-10-14 20:25:34|2026-11-04 17:13:12.761774|20.9    |45  |true |9.0 |565         |
|9045371189|2026-10-14 21:08:54|2026-11-05 16:11:23.051392|21.8    |45  |true |12.5|495         |
+----------+-------------------+--------------------------+--------+----+-----+----+------------+

--- сортировка по дате ЗАГРУЗКИ ---


+----------+-------------------+--------------------------+--------+----+-----+----+------------+
|match_id  |played_at          |sys_insert_stamp          |lag_days|rank|win  |kda |gold_per_min|
+----------+-------------------+--------------------------+--------+----+-----+----+------------+
|9044763059|2026-10-14 14:19:59|2026-10-20 14:17:49.057684|6.0     |15  |true |24.0|510         |
|9045210564|2026-10-14 18:55:24|2026-11-02 00:44:28.869242|18.2    |45  |true |2.67|680         |
|9045280878|2026-10-14 19:46:39|2026-11-03 04:17:24.719599|19.4    |45  |false|2.14|340         |
|9045327747|2026-10-14 20:25:34|2026-11-04 17:13:12.761774|20.9    |45  |true |9.0 |565         |
|9045371189|2026-10-14 21:08:54|2026-11-05 16:11:23.051392|21.8    |45  |true |12.5|495         |
+----------+-------------------+--------------------------+--------+----+-----+----+------------+



### Обоснование farm_role: прокси позиции через net_worth

Готового поля позиции в RAW нет. Проверяем, насколько ранг по `net_worth` внутри команды согласуется с априорными ролями героя из справочника `pg__heroes`. Это обоснование использования `farm_role` для нормировки.

**Выводы:**
* Справочник `pg__heroes` содержит поле `roles` типа `array<string>` — априорные роли героя (Carry, Support, Nuker, Initiator и др.), а также `primary_attr` и `attack_type`.
* **Монотонный градиент подтверждает валидность прокси:**

| farm_role | доля Carry-героев | доля Support-героев | avg GPM |
|---|---|---|---|
| 1 | 92.3% | 12.7% | 655.9 |
| 2 | 81.0% | 21.4% | 549.8 |
| 3 | 67.9% | 27.7% | 472.0 |
| 4 | 32.6% | 63.0% | 378.5 |
| 5 | 18.5% | 75.5% | 324.6 |

* Согласование на крайних позициях высокое: 92.3% Carry на первой и 75.5% Support на пятой. GPM падает почти вдвое от роли 1 к роли 5.
* **Топ героев по farm_role = 1:** Juggernaut, Anti-Mage, Luna, Phantom Lancer, Lifestealer, Phantom Assassin — классические керри.
* **Топ героев по farm_role = 5:** Lion, Shadow Shaman, Pudge, Witch Doctor, Crystal Maiden, Rubick — классические саппорты.
* Число строк по всем пяти ролям почти одинаково (234 616–235 154) — следствие того, что `RANK()` присваивает каждому матчу ровно по одному игроку каждой позиции.
* **Распределение по медалям:** medal 1 (Herald) присутствует — 25 809 строк / 19 155 аккаунтов. Medal 5 (Legend) крупнейший: 308 197 строк. NULL-ранг — 18 241 строка даже в полностью открытых матчах (частичная незаполненность API).
* **Ограничение прокси:** `farm_role` вычисляется по фактическому net_worth, поэтому смурф-саппорт, перфармивший всю команду, попадёт в «керри». Это частичная циркулярность признака, которую необходимо зафиксировать в ограничениях исследования.
* **Решение:** использовать `farm_role` как основу для нормировки признаков доминирования вместо невалидного `rank`.

In [6]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext('hero-role-proxy') as spark:
    h = spark.table("lakehouse.raw.pg__heroes")
    print("=== схема heroes ===")
    h.printSchema()
    h.select("hero_id", "localized_name", "primary_attr", "attack_type", "roles").show(15, truncate=False)

    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")
    base = mp.join(m.select("match_id", "duration"), "match_id")

    cov = base.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"))
    clean = base.join(cov.filter(F.col("named_cnt") == 10).select("match_id"), "match_id")

    with_role = (clean
        .withColumn("farm_role", F.rank().over(
            Window.partitionBy("match_id", "is_radiant").orderBy(F.desc("net_worth"))))
        .join(h.select("hero_id", "localized_name", "roles"), "hero_id"))

    print("=== как роли героя соотносятся с farm_role ===")
    (with_role
        .withColumn("is_carry_hero", F.array_contains("roles", "Carry"))
        .withColumn("is_support_hero", F.array_contains("roles", "Support"))
        .groupBy("farm_role")
        .agg(F.count("*").alias("rows"),
             F.round(F.avg(F.col("is_carry_hero").cast("int")), 3).alias("share_carry_hero"),
             F.round(F.avg(F.col("is_support_hero").cast("int")), 3).alias("share_support_hero"),
             F.round(F.avg("gold_per_min"), 1).alias("gpm"))
        .orderBy("farm_role").show(truncate=False))

    print("=== топ героев по farm_role=1 и =5 ===")
    for r in [1, 5]:
        (with_role.filter(F.col("farm_role") == r)
            .groupBy("localized_name").count()
            .orderBy(F.desc("count")).show(12, truncate=False))

    print("=== есть ли Herald (ранги 1-10) в выборке ===")
    (clean.withColumn("medal", F.floor((F.col("rank") - 1) / 10))
        .groupBy("medal").agg(F.count("*").alias("rows"),
                              F.countDistinct("account_id").alias("accounts"))
        .orderBy("medal").show(truncate=False))

=== схема heroes ===
root
 |-- attack_type: string (nullable = true)
 |-- hero_id: integer (nullable = true)
 |-- localized_name: string (nullable = true)
 |-- name: string (nullable = true)
 |-- primary_attr: string (nullable = true)
 |-- roles: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- job_id: string (nullable = true)
 |-- sys_insert_stamp: timestamp (nullable = true)
 |-- src_modified_stamp: timestamp (nullable = true)



+-------+--------------+------------+-----------+-----------------------------------------+
|hero_id|localized_name|primary_attr|attack_type|roles                                    |
+-------+--------------+------------+-----------+-----------------------------------------+
|1      |Anti-Mage     |agi         |Melee      |[Carry, Escape, Nuker]                   |
|2      |Axe           |str         |Melee      |[Initiator, Durable, Disabler, Carry]    |
|3      |Bane          |all         |Ranged     |[Support, Disabler, Nuker, Durable]      |
|4      |Bloodseeker   |agi         |Melee      |[Carry, Disabler, Nuker, Initiator]      |
|5      |Crystal Maiden|int         |Ranged     |[Support, Disabler, Nuker]               |
|6      |Drow Ranger   |agi         |Ranged     |[Carry, Disabler, Pusher]                |
|7      |Earthshaker   |str         |Melee      |[Support, Initiator, Disabler, Nuker]    |
|8      |Juggernaut    |agi         |Melee      |[Carry, Pusher, Escape]        

+---------+------+----------------+------------------+-----+
|farm_role|rows  |share_carry_hero|share_support_hero|gpm  |
+---------+------+----------------+------------------+-----+
|1        |235028|0.923           |0.127             |655.9|
|2        |235112|0.81            |0.214             |549.8|
|3        |235154|0.679           |0.277             |472.0|
|4        |235100|0.326           |0.63              |378.5|
|5        |234616|0.185           |0.755             |324.6|
+---------+------+----------------+------------------+-----+

=== топ героев по farm_role=1 и =5 ===


26/11/14 04:46:11 WARN ThrowableSerializationWrapper: Task exception could not be deserialized
java.io.InvalidClassException: software.amazon.awssdk.core.exception.SdkClientException; local class incompatible: stream classdesc serialVersionUID = -6298349000205997278, local class serialVersionUID = 4864562373978985526
	at java.base/java.io.ObjectStreamClass.initNonProxy(ObjectStreamClass.java:597)
	at java.base/java.io.ObjectInputStream.readNonProxyDesc(ObjectInputStream.java:2051)
	at java.base/java.io.ObjectInputStream.readClassDesc(ObjectInputStream.java:1898)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2224)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1733)
	at java.base/java.io.ObjectInputStream.readObject(ObjectInputStream.java:509)
	at java.base/java.io.ObjectInputStream.readObject(ObjectInputStream.java:467)
	at java.base/java.util.ArrayList.readObject(ArrayList.java:899)
	at java.base/jdk.internal.reflect.Nati

+----------------+-----+
|localized_name  |count|
+----------------+-----+
|Juggernaut      |15233|
|Anti-Mage       |14153|
|Luna            |11473|
|Phantom Lancer  |10191|
|Lifestealer     |10050|
|Phantom Assassin|9783 |
|Sven            |8738 |
|Sniper          |7597 |
|Necrophos       |6905 |
|Spectre         |5613 |
|Nature's Prophet|5471 |
|Ursa            |5325 |
+----------------+-----+
only showing top 12 rows


+------------------+-----+
|localized_name    |count|
+------------------+-----+
|Lion              |18866|
|Shadow Shaman     |12505|
|Pudge             |11756|
|Witch Doctor      |11204|
|Crystal Maiden    |9658 |
|Rubick            |8312 |
|Lich              |7638 |
|Disruptor         |7370 |
|Skywrath Mage     |6668 |
|Spirit Breaker    |6314 |
|Hoodwink          |5789 |
|Ancient Apparition|5533 |
+------------------+-----+
only showing top 12 rows
=== есть ли Herald (ранги 1-10) в выборке ===


+-----+------+--------+
|medal|rows  |accounts|
+-----+------+--------+
|NULL |18241 |12912   |
|1    |25809 |19155   |
|2    |82361 |59237   |
|3    |151582|103745  |
|4    |250016|163083  |
|5    |308197|191232  |
|6    |178881|108986  |
|7    |159923|91259   |
+-----+------+--------+



### Превышение над эталоном (роль × медаль): вторая версия признаков

Строим превышения `exc_* = z − median(z | farm_role, medal)` — второй уровень нормировки. Проверяем центрированность и отбираем кандидатов с устойчивым превышением при ранге ниже лобби.

**Выводы:**
* Эталон рассчитан для **35 страт** (5 позиций × 7 медалей), размер каждой страты от 4 006 до 63 613 наблюдений — все достаточно наполнены для устойчивой медианы.
* **Эталоны сильно различаются по роли:** для `farm_role = 1` ref_z_dmg ≈ +0.72…+0.82, для `farm_role = 5` ≈ −0.81…−0.83. Разница почти 1.6σ — это и есть эффект позиции, который нужно снять.
* **Внутри роли эталон почти не зависит от медали:** для керри разброс ref_z_dmg по медалям всего 0.717→0.815, для саппорта −0.816→−0.809. Подтверждение вывода из предыдущей ячейки: ранг добавляет мало информации сверх роли.
* Исключение — `farm_role = 2`: ref_z_dmg растёт с 0.085 (medal 1) до 0.442 (medal 7). Здесь медаль действительно значима.
* **Sanity check:** `mean(exc_dmg) = 0.0535`, `std = 0.7769`; `mean(exc_gpm) = 0.0064`, `std = 0.5623`. Признаки приблизительно центрированы (небольшой сдвиг из-за использования медианы вместо среднего и включения строк с NULL-рангом).
* **Кандидаты отобрались:** при фильтрах `n ≥ 3`, `gap < −8`, `exc_dmg > 0.5`, `q25_exc_dmg > 0.2` найдены аккаунты с превышением до 2.32σ. Пример: 337471771 — exc_dmg 2.32, q25 2.27, WR 1.0, 3 героя.
* Фильтр по нижнему квартилю работает как задумано: у большинства кандидатов q25 близок к среднему, что говорит об устойчивости, а не о единичном выбросе.
* **Проблема версии:** признаки всё ещё опираются на `rank` (через `medal` и `gap`), который признан невалидным. Кроме того, эталон по (роль × медаль) не учитывает специфику героя — Meepo и Crystal Maiden на одной позиции получат одинаковый эталон.
* **Следующая итерация:** заменить стратификацию (роль × медаль) на эталон по `hero_id`, а `rank` исключить полностью. Нормировку доминирования оставить на `farm_role`.

In [7]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext('excess-features') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")
    base = mp.join(m.select("match_id", "duration", "start_time"), "match_id")

    cov = base.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"))
    clean = base.join(cov.filter(F.col("named_cnt") == 10).select("match_id"), "match_id")

    w = Window.partitionBy("match_id")
    ctx = (clean
        .withColumn("match_avg_rank", F.avg("rank").over(w))
        .withColumn("match_std_rank", F.stddev_samp("rank").over(w))
        .withColumn("rank_gap", F.col("rank") - F.col("match_avg_rank"))
        .withColumn("z_dmg", (F.col("hero_damage") - F.avg("hero_damage").over(w))
                    / F.nullif(F.stddev_samp("hero_damage").over(w), F.lit(0)))
        .withColumn("z_gpm", (F.col("gold_per_min") - F.avg("gold_per_min").over(w))
                    / F.nullif(F.stddev_samp("gold_per_min").over(w), F.lit(0)))
        .withColumn("z_denies", (F.col("denies") - F.avg("denies").over(w))
                    / F.nullif(F.stddev_samp("denies").over(w), F.lit(0)))
        .withColumn("farm_role", F.rank().over(
            Window.partitionBy("match_id", "is_radiant").orderBy(F.desc("net_worth"))))
        .withColumn("medal", F.floor((F.col("rank") - 1) / 10)))

    # эталон: медиана z для каждой пары (роль, медаль)
    ref = (ctx.filter(F.col("rank").between(1, 80))
              .groupBy("farm_role", "medal")
              .agg(F.expr("percentile_approx(z_dmg, 0.5)").alias("ref_z_dmg"),
                   F.expr("percentile_approx(z_gpm, 0.5)").alias("ref_z_gpm"),
                   F.expr("percentile_approx(z_denies, 0.5)").alias("ref_z_denies"),
                   F.count("*").alias("ref_n")))
    ref.show(40, truncate=False)

    exc = (ctx.join(ref, ["farm_role", "medal"], "left")
              .withColumn("exc_dmg", F.col("z_dmg") - F.col("ref_z_dmg"))
              .withColumn("exc_gpm", F.col("z_gpm") - F.col("ref_z_gpm"))
              .withColumn("exc_denies", F.col("z_denies") - F.col("ref_z_denies")))

    print("=== sanity: excess должен быть центрирован ===")
    exc.agg(F.round(F.avg("exc_dmg"), 4).alias("mean_exc_dmg"),
            F.round(F.stddev("exc_dmg"), 4).alias("std_exc_dmg"),
            F.round(F.avg("exc_gpm"), 4).alias("mean_exc_gpm"),
            F.round(F.stddev("exc_gpm"), 4).alias("std_exc_gpm")).show(truncate=False)

    print("=== кандидаты по excess + низкий ранг относительно лобби ===")
    (exc.filter(F.col("rank").between(1, 80))
        .groupBy("account_id")
        .agg(F.count("*").alias("n"),
             F.round(F.avg("rank"), 1).alias("my_rank"),
             F.round(F.avg("rank_gap"), 1).alias("gap"),
             F.round(F.avg("match_std_rank"), 1).alias("lobby_spread"),
             F.round(F.avg("exc_dmg"), 2).alias("exc_dmg"),
             F.round(F.avg("exc_gpm"), 2).alias("exc_gpm"),
             F.round(F.avg("exc_denies"), 2).alias("exc_denies"),
             F.round(F.percentile_approx("exc_dmg", 0.25), 2).alias("q25_exc_dmg"),
             F.round(F.avg(F.col("win").cast("int")), 2).alias("wr"),
             F.countDistinct("hero_id").alias("heroes"))
        .filter((F.col("n") >= 3) & (F.col("gap") < -8) &
                (F.col("exc_dmg") > 0.5) & (F.col("q25_exc_dmg") > 0.2))
        .orderBy(F.desc("exc_dmg")).show(30, truncate=False))

+---------+-----+---------------------+---------------------+---------------------+-----+
|farm_role|medal|ref_z_dmg            |ref_z_gpm            |ref_z_denies         |ref_n|
+---------+-----+---------------------+---------------------+---------------------+-----+
|1        |7    |0.7880985375021071   |1.3003899501132488   |0.5202456795760882   |35174|
|2        |4    |0.2977730905925871   |0.4453477076699468   |0.19497562956936068  |48697|
|4        |5    |-0.4940177064630053  |-0.6655044842150549  |-0.6554401283039927  |60636|
|5        |4    |-0.8311123857717817  |-1.0263022889301414  |-0.7375837923560066  |51711|
|4        |4    |-0.5101664909698956  |-0.6738673066240304  |-0.6357072528610996  |51402|
|4        |6    |-0.4849521837902691  |-0.6727206475614793  |-0.6687762430634061  |33686|
|2        |2    |0.19477158859044758  |0.3910193958437864   |0.0740797197487193   |15363|
|3        |2    |-0.16127249138678573 |-0.13891684022574433 |-0.2289925202190896  |16682|
|1        

+------------+-----------+------------+-----------+
|mean_exc_dmg|std_exc_dmg|mean_exc_gpm|std_exc_gpm|
+------------+-----------+------------+-----------+
|0.0535      |0.7769     |0.0064      |0.5623     |
+------------+-----------+------------+-----------+

=== кандидаты по excess + низкий ранг относительно лобби ===


+----------+---+-------+-----+------------+-------+-------+----------+-----------+----+------+
|account_id|n  |my_rank|gap  |lobby_spread|exc_dmg|exc_gpm|exc_denies|q25_exc_dmg|wr  |heroes|
+----------+---+-------+-----+------------+-------+-------+----------+-----------+----+------+
|337471771 |3  |11.0   |-14.3|6.3         |2.32   |1.45   |1.05      |2.27       |1.0 |3     |
|834663701 |3  |41.0   |-11.9|4.9         |2.11   |0.75   |0.42      |2.05       |1.0 |2     |
|834661169 |3  |41.0   |-13.6|9.5         |2.0    |1.08   |0.93      |1.62       |1.0 |3     |
|689764739 |3  |34.0   |-8.4 |5.8         |1.81   |1.24   |1.64      |1.69       |1.0 |1     |
|1561758888|4  |50.3   |-10.5|10.9        |1.79   |0.95   |1.13      |1.31       |1.0 |4     |
|712858291 |5  |11.0   |-16.5|7.0         |1.78   |1.15   |0.95      |1.84       |1.0 |5     |
|1906070464|4  |23.0   |-19.3|11.2        |1.77   |0.76   |0.46      |1.29       |0.75|2     |
|421877923 |3  |41.0   |-13.7|12.2        |1.73   

### Природа «растущих» аккаунтов: подъём, артефакт или пропущенные матчи?

Проверяем альтернативное объяснение скачков ранга: возможно, часть матчей аккаунта не попала в выборку (другие режимы, короткие игры), и рост объясняется ненаблюдаемыми победами. Делим growers на группы по винрейту и смотрим полноту покрытия их истории.

**Выводы:**
* Из 173 аккаунтов с ростом ≥15 пунктов: **climber** (WR ≥ 0.6) — 90, **suspicious_growth** (WR ≤ 0.35) — **38**, middle — 45.
* **Покрытие матчей практически полное:** медиана и p90 доли матчей внутри выборки = **1.0**. В среднем 5.7 матча всего, 5.6 из них в выборке. По группам: climber 0.988, suspicious_growth 0.983, middle 0.984.
* **Альтернативная гипотеза отвергнута:** «пропущенных» матчей почти нет. Рост ранга нельзя объяснить ненаблюдаемыми победами в других режимах — вся история аккаунта видна.
* **Хронология подозрительного аккаунта 833431650 (все режимы):** 6 матчей, **все проиграны** (`win = false` во всех строках). KDA крайне низкий: 1/9/0, 0/5/0, 1/12/0, 3/8/2, 0/8/0, 0/4/2. GPM 161–263 при норме ~480. Длительности 940–1438 с — заметно короче типичных 40 минут.
* При этом ранг растёт **13 → 33 → 41** (+28 пунктов, почти 3 медали). Подъём на 3 медали при 0% побед физически невозможен.
* **Для сравнения — climber 799781180:** 5 матчей, все выиграны, KDA 19/4/19, 34/5/19, 16/4/19, 26/2/20, 31/7/19, GPM 584–906. Ранг 13 → 44 → 52. Формально выглядит как реальный подъём, но предыдущая проверка показала: скачок +31 между матчами, сыгранными через 3.5 часа, объясняется датой загрузки записи, а не прогрессом.
* **Итог:** ни одна из групп не даёт валидного сигнала о смене владельца аккаунта. У «честных» climber рост синхронен с `sys_insert_stamp`, у suspicious_growth ранг растёт при total-проигрышах. Признак `rank` непригоден для детекции передачи аккаунта.

In [9]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('growers-nature') as spark:
    m_all = spark.table("lakehouse.raw.pg__matches").select(
        "match_id", "game_mode", "lobby_type", "start_time", "duration", "human_players")

    mp = spark.table("lakehouse.raw.pg__match_players").filter(
        F.col("account_id").isNotNull() & ~F.col("account_id").isin(0, 4294967295))

    m_clean = m_all.filter((F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
                           (F.col("human_players") == 10) & (F.col("duration") >= 720))

    j_clean = (mp.join(m_clean.select("match_id", "start_time"), "match_id")
                 .filter(F.col("rank").between(1, 80)))

    acc = j_clean.groupBy("account_id").agg(
        F.count("*").alias("n"),
        F.min("rank").alias("rmin"),
        F.max("rank").alias("rmax"),
        F.avg(F.col("win").cast("int")).alias("wr"),
        F.avg("kda").alias("kda"))

    growers = acc.filter((F.col("n") >= 4) & ((F.col("rmax") - F.col("rmin")) >= 15))

    print("=== growers: подъём против странного роста ===")
    (growers
        .withColumn("jump", F.col("rmax") - F.col("rmin"))
        .withColumn("kind", F.when(F.col("wr") >= 0.6, F.lit("climber"))
                            .when(F.col("wr") <= 0.35, F.lit("suspicious_growth"))
                            .otherwise(F.lit("middle")))
        .groupBy("kind")
        .agg(F.count("*").alias("accounts"),
             F.round(F.avg("wr"), 3).alias("wr"),
             F.round(F.avg("kda"), 2).alias("kda"),
             F.round(F.avg("jump"), 1).alias("avg_jump"),
             F.round(F.avg("n"), 1).alias("avg_n"))
        .show(truncate=False))

    # все матчи этих аккаунтов во всех режимах
    g_ids = growers.select("account_id")
    all_games = (mp.join(g_ids, "account_id")
                   .join(m_all, "match_id"))

    print("=== покрытие: сколько матчей вне нашей выборки ===")
    coverage = (all_games
        .withColumn("in_sample",
                    ((F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
                     (F.col("duration") >= 720)).cast("int"))
        .groupBy("account_id")
        .agg(F.count("*").alias("games_total"),
             F.sum("in_sample").alias("games_in_sample"),
             F.countDistinct("game_mode").alias("modes"))
        .withColumn("cov", F.col("games_in_sample") / F.col("games_total")))

    coverage.selectExpr(
        "count(*) as accounts",
        "percentile_approx(cov, array(0.1,0.5,0.9)) as coverage_q",
        "round(avg(games_total),1) as avg_total",
        "round(avg(games_in_sample),1) as avg_in_sample"
    ).show(truncate=False)

    print("=== покрытие в разрезе kind ===")
    (coverage.join(growers.select("account_id", "wr", "kda"), "account_id")
        .withColumn("kind", F.when(F.col("wr") >= 0.6, F.lit("climber"))
                            .when(F.col("wr") <= 0.35, F.lit("suspicious_growth"))
                            .otherwise(F.lit("middle")))
        .groupBy("kind")
        .agg(F.count("*").alias("accounts"),
             F.round(F.avg("cov"), 3).alias("avg_coverage"),
             F.round(F.avg("games_total"), 1).alias("games_total"),
             F.round(F.avg("games_in_sample"), 1).alias("games_in_sample"))
        .show(truncate=False))

    print("=== хронология подозрительного аккаунта (все режимы) ===")
    (all_games.filter(F.col("account_id") == 833431650)
        .select(F.from_unixtime("start_time").alias("played_at"), "match_id",
                "game_mode", "lobby_type", "duration", "rank", "hero_id",
                "kills", "deaths", "assists", "gold_per_min", "win")
        .orderBy("played_at").show(40, truncate=False))

    print("=== хронология честного climber для сравнения ===")
    (all_games.filter(F.col("account_id") == 799781180)
        .select(F.from_unixtime("start_time").alias("played_at"), "match_id",
                "game_mode", "lobby_type", "rank", "hero_id",
                "kills", "deaths", "assists", "gold_per_min", "win")
        .orderBy("played_at").show(40, truncate=False))

=== growers: подъём против странного роста ===


+-----------------+--------+-----+----+--------+-----+
|kind             |accounts|wr   |kda |avg_jump|avg_n|
+-----------------+--------+-----+----+--------+-----+
|suspicious_growth|38      |0.117|1.69|18.6    |6.0  |
|climber          |90      |0.817|7.95|19.3    |5.0  |
|middle           |45      |0.46 |4.72|17.8    |5.3  |
+-----------------+--------+-----+----+--------+-----+

=== покрытие: сколько матчей вне нашей выборки ===


26/11/14 04:58:18 WARN S3FileIO: Unclosed S3FileIO instance created by:
	org.apache.iceberg.aws.s3.S3FileIO.initialize(S3FileIO.java:507)
	org.apache.iceberg.CatalogUtil.loadFileIO(CatalogUtil.java:423)
	org.apache.iceberg.rest.RESTSessionCatalog.newFileIO(RESTSessionCatalog.java:1201)
	org.apache.iceberg.rest.RESTSessionCatalog.tableFileIO(RESTSessionCatalog.java:1219)
	org.apache.iceberg.rest.RESTSessionCatalog.lambda$createTableSupplier$3(RESTSessionCatalog.java:576)
	org.apache.iceberg.rest.RESTSessionCatalog.loadTable(RESTSessionCatalog.java:559)
	org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.loadTable(BaseSessionCatalog.java:105)
	org.apache.iceberg.rest.RESTCatalog.loadTable(RESTCatalog.java:124)
	org.apache.iceberg.shaded.com.github.benmanes.caffeine.cache.BoundedLocalCache.lambda$doComputeIfAbsent$14(BoundedLocalCache.java:2406)
	java.base/java.util.concurrent.ConcurrentHashMap.compute(ConcurrentHashMap.java:1916)
	org.apache.iceberg.shaded.com.github.benmanes.caffei

+--------+---------------+---------+-------------+
|accounts|coverage_q     |avg_total|avg_in_sample|
+--------+---------------+---------+-------------+
|173     |[1.0, 1.0, 1.0]|5.7      |5.6          |
+--------+---------------+---------+-------------+

=== покрытие в разрезе kind ===


+-----------------+--------+------------+-----------+---------------+
|kind             |accounts|avg_coverage|games_total|games_in_sample|
+-----------------+--------+------------+-----------+---------------+
|suspicious_growth|38      |0.983       |6.7        |6.5            |
|climber          |90      |0.988       |5.4        |5.3            |
|middle           |45      |0.984       |5.6        |5.5            |
+-----------------+--------+------------+-----------+---------------+

=== хронология подозрительного аккаунта (все режимы) ===


+-------------------+----------+---------+----------+--------+----+-------+-----+------+-------+------------+-----+
|played_at          |match_id  |game_mode|lobby_type|duration|rank|hero_id|kills|deaths|assists|gold_per_min|win  |
+-------------------+----------+---------+----------+--------+----+-------+-----+------+-------+------------+-----+
|2026-10-14 14:55:00|9044823341|22       |7         |944     |13  |45     |1    |9     |0      |163         |false|
|2026-10-14 17:07:25|9045042889|22       |7         |948     |13  |16     |0    |5     |0      |161         |false|
|2026-10-14 20:36:00|9045339019|22       |7         |1363    |33  |76     |1    |12    |0      |202         |false|
|2026-10-14 21:03:48|9045366406|22       |7         |940     |33  |79     |3    |8     |2      |263         |false|
|2026-10-14 21:27:34|9045388020|22       |7         |1438    |33  |58     |0    |8     |0      |197         |false|
|2026-10-15 02:27:20|9045567180|22       |7         |1107    |41  |58   

+-------------------+----------+---------+----------+----+-------+-----+------+-------+------------+----+
|played_at          |match_id  |game_mode|lobby_type|rank|hero_id|kills|deaths|assists|gold_per_min|win |
+-------------------+----------+---------+----------+----+-------+-----+------+-------+------------+----+
|2026-10-14 13:34:45|9044686318|22       |7         |13  |86     |19   |4     |19     |642         |true|
|2026-10-14 15:09:05|9044847533|22       |7         |13  |38     |34   |5     |19     |906         |true|
|2026-10-14 18:38:39|9045185347|22       |7         |44  |110    |16   |4     |19     |584         |true|
|2026-10-14 19:28:04|9045256223|22       |7         |52  |86     |26   |2     |20     |723         |true|
|2026-10-14 20:11:19|9045311324|22       |7         |52  |38     |31   |7     |19     |862         |true|
+-------------------+----------+---------+----------+----+-------+-----+------+-------+------------+----+



### Кодировка поля rank: какие значения реально присутствуют

Уточняем структуру ранга: точные значения, границы диапазона, наличие пиков. Нужно для корректного построения медалей и понимания, что означает NULL внутри полностью открытых матчей.

**Выводы:**
* **Диапазон значений: 11–80, всего 36 уникальных значений.** Ранги 1–10 (Herald) в выборке отсутствуют полностью. Минимальное наблюдаемое значение — 11 (Guardian I), встречается 264 раза.
* Значения распределены неравномерно внутри декад: 11 → 264, 12 → 3 680, 13 → 5 302, 14 → 7 237, 15 → 9 326; далее сразу 21 → 11 643. Промежуточных значений (16–20) нет — кодировка соответствует `medal × 10 + star`, где звезда принимает значения 1–5.
* **Пики:** самые частые значения — 52 (64 389), 53 (64 200), 51 (62 533), 54 (61 933), **80 (61 198)**. Значение 80 — Immortal, верхняя граница без деления на звёзды, поэтому аккумулирует всех игроков высшего дивизиона.
* Мода приходится на диапазон 51–55 (Legend), что согласуется с медианой среднего ранга матча ≈ 50 из предыдущих проверок.
* **NULL-ранг внутри полностью открытых матчей:** 18 241 строка / 12 912 аккаунтов (1.55% от 1 175 010). Это игроки, скрывшие ранг отдельно от профиля, либо незаполненные записи API.
* **Игроки с NULL-рангом систематически слабее:** GPM 416.8 против 477.2 (**−12.7%**), KDA 4.08 против 4.31, denies 5.3 против 7.0 (**−24%**), WR 0.488 против 0.500.
* Это важное наблюдение: пропуски ранга **не случайны** (MNAR — missing not at random). Заполнение средним или исключение таких строк внесло бы смещение. Дополнительный аргумент в пользу полного отказа от `rank` как признака.
* Отсутствие Herald означает, что выборка не покрывает нижний дивизион. Для детекции смурфов это ограничение: типичный смурф играет именно в низких рангах, где контраст со его реальным уровнем максимален.

In [10]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext('rank-coding') as spark:
    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")
    base = mp.join(m.select("match_id"), "match_id")

    cov = base.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"))
    clean = base.join(cov.filter(F.col("named_cnt") == 10).select("match_id"), "match_id")

    print("=== точные значения rank <= 25 ===")
    (clean.filter(F.col("rank") <= 25).groupBy("rank").count()
        .orderBy("rank").show(30, truncate=False))

    print("=== минимум / максимум / число уникальных ===")
    clean.agg(F.min("rank").alias("rmin"), F.max("rank").alias("rmax"),
              F.countDistinct("rank").alias("distinct_ranks")).show(truncate=False)

    print("=== игроки с NULL rank внутри открытых матчей ===")
    (clean.withColumn("has_rank", F.col("rank").isNotNull().cast("int"))
        .groupBy("has_rank")
        .agg(F.count("*").alias("rows_cnt"),
             F.countDistinct("account_id").alias("accounts"),
             F.round(F.avg("gold_per_min"), 1).alias("gpm"),
             F.round(F.avg("kda"), 2).alias("kda"),
             F.round(F.avg("denies"), 1).alias("denies"),
             F.round(F.avg(F.col("win").cast("int")), 3).alias("wr"))
        .show(truncate=False))

    print("=== пики по конкретным значениям ранга ===")
    (clean.filter(F.col("rank").isNotNull()).groupBy("rank").count()
        .orderBy(F.desc("count")).show(15, truncate=False))

=== точные значения rank <= 25 ===


+----+-----+
|rank|count|
+----+-----+
|11  |264  |
|12  |3680 |
|13  |5302 |
|14  |7237 |
|15  |9326 |
|21  |11643|
|22  |14245|
|23  |16964|
|24  |18627|
|25  |20882|
+----+-----+

=== минимум / максимум / число уникальных ===


+----+----+--------------+
|rmin|rmax|distinct_ranks|
+----+----+--------------+
|11  |80  |36            |
+----+----+--------------+

=== игроки с NULL rank внутри открытых матчей ===


+--------+--------+--------+-----+----+------+-----+
|has_rank|rows_cnt|accounts|gpm  |kda |denies|wr   |
+--------+--------+--------+-----+----+------+-----+
|1       |1156769 |714201  |477.2|4.31|7.0   |0.5  |
|0       |18241   |12912   |416.8|4.08|5.3   |0.488|
+--------+--------+--------+-----+----+------+-----+

=== пики по конкретным значениям ранга ===


+----+-----+
|rank|count|
+----+-----+
|52  |64389|
|53  |64200|
|51  |62533|
|54  |61933|
|80  |61198|
|45  |57190|
|55  |55142|
|44  |53699|
|43  |50278|
|61  |48418|
|42  |46155|
|41  |42694|
|62  |40656|
|35  |36905|
|63  |34936|
+----+-----+
only showing top 15 rows


### Версия v3: центрирование средним, нормированный gap и признак эффективности золота

Третья итерация признаков. Изменения относительно v2: эталон считается как **среднее** (не медиана) для точного центрирования, `gap` нормируется на std ранга лобби с защитой от деления на малое значение, добавлены признаки `exc_dpm` и `exc_gu` (непотраченное золото), а также роль героя из справочника.

**Выводы:**
* **Центрирование работает точно:** `mean(exc_dmg) = 0.0`, `mean(exc_gpm) = 0.0`, `mean(exc_den) = 0.0`. Std: 0.776 / 0.559 соответственно. Переход от медианы к среднему устранил сдвиг +0.05, наблюдавшийся в v2.
* **Найдена интересная группа — саппорт-герои на позициях 1–2 по фарму.** Все 20 показанных аккаунтов имеют WR = 1.0 при `exc_gpm` 1.03–1.33 и `exc_dmg` до 2.12. Пример: 1419729683 (exc_gpm 1.08, exc_dmg 2.12), 926332459 (rank 80, exc_gpm 1.11, exc_dmg 1.98). Это либо смурфы на нестандартных ролях, либо игроки с unusual стилем — требует ручного разбора.
* Признак `hero_role` из справочника (`Carry` / `Support` / `flex`) позволяет находить такие случаи: расхождение априорной роли героя и фактической позиции по net_worth.
* **Обнаружен критический баг в `gold_unused`.** Формула `1 − gold_spent / gold` даёт значения **от −1455.514 до +44.421** вместо ожидаемого диапазона [0, 1]. Примеры: аккаунт 100173511 → exc_gu = −1455.514; 190472894 → −1239.379; большинство остальных → +32…+44.
* **Причина:** семантика полей неверно понята. `gold` — это **остаток золота на конец матча** (медиана ~1330), а `gold_spent` — потраченное за игру (медиана ~15 985). Они несопоставимы: деление большего на меньшее даёт огромные отрицательные значения, а когда остаток велик — значения около +1, которые после вычитания эталона превращаются в +40.
* **Правильная формула:** `gold_leftover_share = gold / (gpm × duration_min)` — доля заработанного золота, оставшаяся непотраченной. Проверка семантики полей выполнена отдельным запросом (см. ноутбук валидации данных).
* **Итоговые кандидаты v3** отобраны по `n ≥ 3`, `gap_norm < −1.5`, `exc_dmg > 0.6`, `q25_exc_dmg > 0.3`. Топ: 337471771 (exc_dmg 2.22, q25 2.17, 3 героя, 1 роль, WR 1.0), 834663701 (2.04 / 1.99), 712858291 (1.76 / 1.82, 5 героев).
* Фильтр по нижнему квартилю подтверждает устойчивость: у большинства кандидатов q25 близок к среднему (2.22 vs 2.17; 2.04 vs 1.99; 1.76 vs 1.82).
* **Версия не принята в production** по двум причинам: (1) баг в `gold_unused`; (2) все фильтры опираются на `rank` через `gap_norm` и `medal`, который признан невалидным снапшотом. Следующая итерация — отказ от ранга, эталон по `hero_id`, исправленная формула золота.

In [11]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window

with SparkSessionContext('features-v3') as spark:
    h = spark.table("lakehouse.raw.pg__heroes").select(
        "hero_id", "localized_name",
        F.array_contains(F.col("roles"), F.lit("Carry")).alias("hero_carry"),
        F.array_contains(F.col("roles"), F.lit("Support")).alias("hero_support"))

    m = spark.table("lakehouse.raw.pg__matches").filter(
        (F.col("game_mode") == 22) & (F.col("lobby_type") == 7) &
        (F.col("human_players") == 10) & (F.col("duration") >= 720))
    mp = spark.table("lakehouse.raw.pg__match_players")
    base = mp.join(m.select("match_id", "duration", "start_time"), "match_id")

    cov = base.groupBy("match_id").agg(
        F.sum((F.col("account_id").isNotNull() &
               ~F.col("account_id").isin(0, 4294967295)).cast("int")).alias("named_cnt"))
    clean = (base.join(cov.filter(F.col("named_cnt") == 10).select("match_id"), "match_id")
                 .join(h, "hero_id"))

    w = Window.partitionBy("match_id")
    wt = Window.partitionBy("match_id", "is_radiant")

    ctx = (clean
        .withColumn("m_rank", F.avg("rank").over(w))
        .withColumn("s_rank", F.stddev_samp("rank").over(w))
        .withColumn("gap_norm", (F.col("rank") - F.col("m_rank"))
                                / F.greatest(F.col("s_rank"), F.lit(3.0)))
        .withColumn("z_dmg", (F.col("hero_damage") - F.avg("hero_damage").over(w))
                    / F.nullif(F.stddev_samp("hero_damage").over(w), F.lit(0)))
        .withColumn("z_gpm", (F.col("gold_per_min") - F.avg("gold_per_min").over(w))
                    / F.nullif(F.stddev_samp("gold_per_min").over(w), F.lit(0)))
        .withColumn("z_den", (F.col("denies") - F.avg("denies").over(w))
                    / F.nullif(F.stddev_samp("denies").over(w), F.lit(0)))
        .withColumn("farm_role", F.rank().over(wt.orderBy(F.desc("net_worth"))))
        .withColumn("hero_role",
                    F.when(F.col("hero_carry") & ~F.col("hero_support"), F.lit("carry"))
                     .when(F.col("hero_support") & ~F.col("hero_carry"), F.lit("support"))
                     .otherwise(F.lit("flex")))
        .withColumn("medal", F.floor((F.col("rank") - F.lit(1)) / F.lit(10)))
        .withColumn("dpm", F.col("denies") / (F.col("duration") / F.lit(60.0)))
        .withColumn("gold_unused", F.lit(1) - F.col("gold_spent")
                    / F.nullif(F.col("gold"), F.lit(0))))

    ref = (ctx.filter(F.col("rank").isNotNull())
              .groupBy("farm_role", "medal")
              .agg(F.avg("z_dmg").alias("ref_dmg"),
                   F.avg("z_gpm").alias("ref_gpm"),
                   F.avg("z_den").alias("ref_den"),
                   F.avg("dpm").alias("ref_dpm"),
                   F.avg("gold_unused").alias("ref_gu"),
                   F.count("*").alias("ref_n")))

    exc = (ctx.join(ref, ["farm_role", "medal"], "left")
              .withColumn("exc_dmg", F.col("z_dmg") - F.col("ref_dmg"))
              .withColumn("exc_gpm", F.col("z_gpm") - F.col("ref_gpm"))
              .withColumn("exc_den", F.col("z_den") - F.col("ref_den"))
              .withColumn("exc_dpm", F.col("dpm") - F.col("ref_dpm"))
              .withColumn("exc_gu", F.col("gold_unused") - F.col("ref_gu")))

    print("=== sanity после центрирования средним ===")
    exc.agg(F.round(F.avg("exc_dmg"), 5).alias("m_dmg"),
            F.round(F.stddev("exc_dmg"), 3).alias("s_dmg"),
            F.round(F.avg("exc_gpm"), 5).alias("m_gpm"),
            F.round(F.stddev("exc_gpm"), 3).alias("s_gpm"),
            F.round(F.avg("exc_den"), 5).alias("m_den")).show(truncate=False)

    print("=== саппорт-герои на первой-второй позиции по фарму ===")
    (exc.filter((F.col("hero_role") == "support") & (F.col("farm_role") <= 2))
        .groupBy("account_id")
        .agg(F.count("*").alias("n"),
             F.round(F.avg("rank"), 1).alias("my_rank"),
             F.round(F.avg("exc_gpm"), 2).alias("exc_gpm"),
             F.round(F.avg("exc_dmg"), 2).alias("exc_dmg"),
             F.round(F.avg(F.col("win").cast("int")), 2).alias("wr"))
        .filter(F.col("n") >= 2)
        .orderBy(F.desc("exc_gpm")).show(20, truncate=False))

    print("=== итоговые кандидаты ===")
    (exc.filter(F.col("rank").isNotNull())
        .groupBy("account_id")
        .agg(F.count("*").alias("n"),
             F.round(F.avg("rank"), 1).alias("my_rank"),
             F.round(F.avg("gap_norm"), 2).alias("gap_norm"),
             F.round(F.avg("exc_dmg"), 2).alias("exc_dmg"),
             F.round(F.percentile_approx(F.col("exc_dmg"), F.lit(0.25)), 2).alias("q25_exc_dmg"),
             F.round(F.avg("exc_den"), 2).alias("exc_den"),
             F.round(F.avg("exc_gu"), 3).alias("exc_gu"),
             F.round(F.avg(F.col("win").cast("int")), 2).alias("wr"),
             F.countDistinct("hero_id").alias("heroes"),
             F.countDistinct("farm_role").alias("roles_played"))
        .filter((F.col("n") >= 3) & (F.col("gap_norm") < -1.5) &
                (F.col("exc_dmg") > 0.6) & (F.col("q25_exc_dmg") > 0.3))
        .orderBy(F.desc("exc_dmg")).show(30, truncate=False))

=== sanity после центрирования средним ===


+-----+-----+-----+-----+-----+
|m_dmg|s_dmg|m_gpm|s_gpm|m_den|
+-----+-----+-----+-----+-----+
|0.0  |0.776|0.0  |0.559|0.0  |
+-----+-----+-----+-----+-----+

=== саппорт-герои на первой-второй позиции по фарму ===


+----------+---+-------+-------+-------+---+
|account_id|n  |my_rank|exc_gpm|exc_dmg|wr |
+----------+---+-------+-------+-------+---+
|1139292565|2  |33.0   |1.33   |1.89   |1.0|
|1197095255|2  |14.0   |1.24   |1.65   |1.0|
|793156747 |2  |14.0   |1.18   |1.71   |1.0|
|1322292494|2  |54.0   |1.18   |1.52   |1.0|
|1074092100|2  |25.0   |1.14   |0.58   |1.0|
|248990937 |2  |32.0   |1.14   |0.84   |1.0|
|196896631 |2  |45.0   |1.14   |0.32   |1.0|
|1720310425|2  |63.0   |1.14   |1.52   |1.0|
|706305839 |2  |53.0   |1.12   |1.83   |1.0|
|926332459 |2  |80.0   |1.11   |1.98   |1.0|
|800397067 |2  |13.0   |1.11   |1.78   |1.0|
|1418669556|2  |18.5   |1.1    |2.0    |1.0|
|1419729683|2  |27.5   |1.08   |2.12   |1.0|
|1839362463|2  |32.0   |1.06   |0.7    |1.0|
|403110617 |3  |25.0   |1.06   |1.39   |1.0|
|330477442 |3  |34.0   |1.05   |1.28   |1.0|
|21901457  |2  |62.0   |1.05   |1.24   |1.0|
|1278098614|2  |72.5   |1.04   |1.78   |1.0|
|1048038685|2  |54.0   |1.03   |1.01   |1.0|
|979911750

[Stage 65:=============================>                            (1 + 1) / 2]

+----------+---+-------+--------+-------+-----------+-------+---------+----+------+------------+
|account_id|n  |my_rank|gap_norm|exc_dmg|q25_exc_dmg|exc_den|exc_gu   |wr  |heroes|roles_played|
+----------+---+-------+--------+-------+-----------+-------+---------+----+------+------------+
|337471771 |3  |11.0   |-2.28   |2.22   |2.17       |0.81   |42.281   |1.0 |3     |1           |
|834663701 |3  |41.0   |-2.46   |2.04   |1.99       |0.25   |39.637   |1.0 |2     |2           |
|712858291 |5  |11.0   |-2.42   |1.76   |1.82       |0.78   |43.86    |1.0 |5     |1           |
|1906070464|4  |23.0   |-1.76   |1.71   |1.21       |0.28   |37.044   |0.75|2     |2           |
|188633913 |3  |43.0   |-1.56   |1.69   |1.56       |0.45   |33.341   |1.0 |3     |1           |
|1609344558|3  |15.0   |-1.78   |1.68   |1.27       |0.3    |38.763   |1.0 |2     |2           |
|126555791 |3  |44.0   |-1.74   |1.68   |1.09       |0.8    |10.516   |1.0 |3     |2           |
|100173511 |3  |72.0   |-1.76 